# Step 0 &nbsp;&middot;&nbsp; Data source validation

**DSCI 510 project — *Which software vulnerabilities actually get attacked?***

Before any analysis, prove that three independent APIs are reachable, that our credentials work,
that we understand their constraints, that their fields mean what we think, and that all three
can be joined into one table. Every check writes a `PASS` / `WARN` / `FAIL` verdict to
`logs/validation_report.json`.

### The three sources

| Source | Role in the project | Auth |
|:--|:--|:--|
| **NVD CVE API 2.0** (NIST) | **Features** — description, weakness class, severity vector, as the CNA published them | API key |
| **CISA KEV catalog** | **Labels** — which CVEs were confirmed exploited in the wild | none |
| **EPSS** (FIRST) | **Benchmark** — the industry-standard exploitation probability | none |

Two supporting services are probed once, to settle open questions: NVD's **CVE Change History API**
and the CVE Program's own **CVE Services API**.

### How to run

1. Put your NVD API key in a file named `.env` beside this notebook: `NVD_API_KEY=your-key-here`
2. Run every cell top to bottom — no cell depends on a later one.
3. A failed step prints `FAIL` and the notebook **keeps going**, so one run gives the full diagnosis.

**Runtime** ≈ 20 minutes &nbsp;&middot;&nbsp; **Writes** `data/raw/`, `data/processed/`, `logs/`, `requirements-lock.txt`

### What each step is actually testing

| Steps | Failure class it rules out | Why it matters |
|:--|:--|:--|
| 5 | **Access** — host unreachable, firewall, VPN | Loud and obvious |
| 3, 6, 6b | **Authorisation** — key missing or silently ignored, and proof the test can fail | Silent: you just get throttled |
| 8, 9, 15 | **Constraints** — page size, 120-day cap, window edges | Discovered mid-pull otherwise |
| 7, 10, 11, 12 | **Semantics** — fields mean what we assume, including *who* produced them | Produces confident, wrong analysis |
| **14** | **Integration** — the three sources actually join, with nothing leaking across | **Project-ending if it fails** |
| 13b, 16, 17 | **Design** — publication-time data, the benchmark's data path, dataset size, and enough positives to train on | Decides the modelling plan |
| 19 | **Drift** — what changed since the previous run, separated into normal growth and changes that matter | Live sources change under you |

<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">WHAT CHANGED IN VERSION 4 (NOVEMBER 10)</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">Re-run before the full download, a month after version 3. Two steps were added: <b>13b</b> downloads EPSS's daily score files, checks that they agree with the API, and records which EPSS model was scoring (the source of every historical score in Q4); <b>19</b> compares this run with the previous one. Step 18 now also keeps a <b>dated copy</b> of the report (<code>logs/validation_report_YYYY-MM-DD.json</code>), so runs can be compared. Nothing in steps 1&ndash;17 changed.</div>
</div>

<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">WHAT CHANGED IN VERSION 3 (OCTOBER 9)</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">Every fix is tagged in the code with its ID from <code>docs/CORRECTIONS_LOG.md</code>. The big ones: severity and weakness now come <b>only from the CVE's own CNA</b>, searched across every CVSS version (A1, A26, A3, B1); <code>days_to_kev</code> counts <b>calendar days</b> (A2); the join brings over <b>only the label</b> from KEV (A35); every check now tests a real condition instead of "it ran" (A19, A20, A30, A32, A37); and new steps measure what the plan depends on (steps 6b, 15&ndash;17).</div>
</div>

---

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 01 &nbsp;&middot;&nbsp; SETUP</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Install dependencies</div>
</div>


**Why** — guarantees the notebook runs on a machine that has never run it before.
`%pip` installs into the *kernel executing this notebook*, which is not always the Python on your PATH.

**Note** — `pyarrow` is now **required**: parquet keeps dates and types, which CSV would lose.

**Pass** — no error. A "restart the kernel" message is normal.

In [1]:
%pip install --quiet requests pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 02 &nbsp;&middot;&nbsp; ENVIRONMENT</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Imports, version check, and the recording mechanism</div>
</div>


**Why** — if this fails, the problem is your Python environment, not the APIs.
pandas 2.0+ is required: the date parsing in step 10 uses `format="ISO8601"`, which pandas 1.x
cannot do.

**Also defined here**

| Name | What it does |
|:--|:--|
| `record(step, ok, **details)` | Prints a verdict *and* stores it in `REPORT`. `ok=None` means **WARN** — it ran, but a human should look. |
| `EXPECTED_STEPS` | The full list of checks. One that never runs is reported as a FAIL at the end, not silently left out. |
| `note_source(name, ...)` | Stamps each source's access time *when its data actually arrives* — a graded requirement. |
| `show_checks({...})` | Prints a checklist of named conditions, so a FAIL says exactly which condition failed. |

**Pass** — version numbers print, then `environment OK`.

In [2]:
import os, sys, time, json, subprocess
import datetime as dt
from collections import Counter
from pathlib import Path
from getpass import getpass
from urllib.parse import urlencode
import requests
import pandas as pd
import numpy as np

print("python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("requests:", requests.__version__)

try:
    import pyarrow
    print("pyarrow :", pyarrow.__version__)
except ImportError as exc:
    # Required since v3: CSV would turn dates back into text and mangle list columns (A38).
    raise RuntimeError("pyarrow is required. Run step 01, restart the kernel, and run again.") from exc

if int(pd.__version__.split(".")[0]) < 2:
    raise RuntimeError(f"pandas {pd.__version__} is too old. Run: pip install -U pandas")

RAW  = Path("data/raw");       RAW.mkdir(parents=True, exist_ok=True)
PROC = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)
LOGS = Path("logs");           LOGS.mkdir(parents=True, exist_ok=True)
PROBES = LOGS / "probes";      PROBES.mkdir(parents=True, exist_ok=True)

def now_iso():
    """Current local time with its UTC offset, e.g. 2026-10-10T09:15:02-07:00."""
    return dt.datetime.now().astimezone().isoformat(timespec="seconds")

# Every check this notebook should record. A step still missing at the end counts as FAIL (A40).
EXPECTED_STEPS = [
    "03-key-loaded", "05-nvd-reachable", "06-key-honoured", "06b-negative-control",
    "07-known-record", "08-date-limit-understood", "09-client-built", "10-parser-selftest",
    "11-nvd-window", "12-kev", "13-epss", "13b-epss-bulk", "14-join", "15-window-boundaries",
    "16-change-history", "16-cve-program-dates", "17-dataset-size", "17-label-feasibility",
    "18-saved", "19-compare-runs",
]
REPORT = {"started": now_iso(), "steps": {}, "sources": {}, "findings": {}}

def record(step, ok, **details):
    """Record one check's outcome, print a one-line verdict, and keep going.
    ok=None means WARN: the check ran, but the result needs a human look."""
    verdict = "WARN" if ok is None else ("PASS" if ok else "FAIL")
    REPORT["steps"][step] = {"ok": verdict == "PASS", "verdict": verdict, **details}
    print(f"{verdict}  {step}" + (f"  |  {details}" if details else ""))

def note_source(name, **details):
    """Stamp a source's access time at the moment its data actually arrived (A39)."""
    REPORT["sources"][name] = {"accessed": now_iso(), **details}

def show_checks(checks):
    """Print a dict of {description: passed} as a checklist; return the failed names."""
    for name, passed in checks.items():
        print(f"  {'ok ' if passed else 'NO '} {name}")
    return [name for name, passed in checks.items() if not passed]

print("\nenvironment OK")

python  : 3.14.6
pandas  : 3.0.3
numpy   : 2.4.6
requests: 2.34.2
pyarrow : 23.0.1

environment OK


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 03 &nbsp;&middot;&nbsp; AUTHORISATION</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Load the API key without leaking it</div>
</div>


**Why** — everything after this needs the credential, and a notebook *saves its output*, so a
printed key would ship with your submission.

**How** — the key is looked for in three places, in order:

| Order | Source | When it is used |
|:--|:--|:--|
| 1 | `NVD_API_KEY` environment variable | If you set one in your shell |
| 2 | `.env` file beside the notebook | **The normal path** — one line: `NVD_API_KEY=...` |
| 3 | Hidden `getpass` prompt | If neither of the above exists |

**Pass** — `source` names which path supplied it, `length` is 36 (NVD keys are UUIDs), and only the
last four characters are shown.

<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">REMINDER</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">Add <code>.env</code> to <code>.gitignore</code> before committing anything.</div>
</div>

In [3]:
def load_api_key(var="NVD_API_KEY", env_file=".env"):
    """Return the key from the environment, else a local .env file, else an interactive prompt."""
    if os.environ.get(var):
        return os.environ[var].strip(), "environment variable"
    path = Path(env_file)
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            name, _, value = line.partition("=")
            if name.strip() == var:
                value = value.strip().strip('"').strip("'")
                if value and "your-key" not in value:
                    return value, f"{env_file} file"
    return getpass(f"{var} (hidden input): ").strip(), "interactive prompt"

NVD_API_KEY, key_source = load_api_key()
ok = bool(NVD_API_KEY) and len(NVD_API_KEY) > 20
record("03-key-loaded", ok, source=key_source, length=len(NVD_API_KEY),
       fingerprint=f"...{NVD_API_KEY[-4:]}" if NVD_API_KEY else None)

if not ok:
    print("\n  The key looks wrong. An NVD key is a 36-character UUID like")
    print("  XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX")

PASS  03-key-loaded  |  {'source': '.env file', 'length': 36, 'fingerprint': '...6A30'}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 04 &nbsp;&middot;&nbsp; PLUMBING</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">One HTTP helper for all three APIs</div>
</div>


**Why** — every network concern belongs in one place: headers, retries, backoff, error translation.

**Things worth noticing in the code**

| Mechanism | What it does |
|:--|:--|
| A descriptive `User-Agent` header | A courtesy that identifies us. No source says it is required. |
| Backoff on 403 / 429 / 500 / 502 / 503 / 504 | Waits 6 s, 12 s, 24 s between attempts; never sleeps after the final one |
| `reason(resp)` | Reads NVD's `message` **header** first — NVD puts its explanations there, with an empty body |
| A custom `ApiError` exception | Later cells catch *our* failures and record them, without swallowing real bugs |
| `query(params, flags)` | Builds query strings that include NVD's value-less flags such as `noRejected` |

**Pass** — prints `helper ready`. Nothing is called yet; a quiet cell is the correct result.

In [4]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "DSCI510-course-project/1.0 (academic research)",   # a courtesy, not a known requirement
    "Accept": "application/json",
})

class ApiError(RuntimeError):
    """A failure we understand. `status` holds the HTTP status code when there was one."""
    def __init__(self, message, status=None):
        super().__init__(message)
        self.status = status

RETRYABLE = {403, 429, 500, 502, 503, 504}     # throttling, or a server that may recover (A5)

def reason(resp):
    """NVD explains refusals in a `message` header with an empty body; others use the body (A22)."""
    return resp.headers.get("message") or resp.text[:200].strip() or "(no reason given)"

def http_get(url, params=None, headers=None, max_attempts=4, pause=1.0, timeout=60):
    """GET with backoff. Raises ApiError carrying a human-readable cause.
    max_attempts counts every try, the first one included (A9)."""
    last = None
    for attempt in range(1, max_attempts + 1):
        final = attempt == max_attempts
        try:
            resp = SESSION.get(url, params=params, headers=headers, timeout=timeout)
        except requests.exceptions.SSLError as exc:
            raise ApiError("TLS failure - usually a corporate proxy or VPN intercepting HTTPS. "
                           "Try another network, or off the university VPN.") from exc
        except requests.exceptions.Timeout:
            # Checked before ConnectionError: a connect-timeout is both, and is worth a retry.
            last = f"no response within {timeout}s"
            print(f"  timeout (attempt {attempt}/{max_attempts})")
            if not final:                                   # never sleep just to give up (A4)
                time.sleep(2 ** attempt)
            continue
        except requests.exceptions.ConnectionError as exc:
            raise ApiError(f"Cannot reach {url}. Check internet access, a firewall, or a VPN "
                           "blocking this host.") from exc
        if resp.status_code == 200:
            time.sleep(pause)
            return resp
        if resp.status_code in RETRYABLE:
            last = f"HTTP {resp.status_code}: {reason(resp)}"
            if not final:
                wait = 6 * 2 ** (attempt - 1)                # 6 s, 12 s, 24 s
                print(f"  {last}; backing off {wait}s (attempt {attempt}/{max_attempts})")
                time.sleep(wait)
            continue
        raise ApiError(f"HTTP {resp.status_code} from {url}: {reason(resp)}", status=resp.status_code)
    raise ApiError(f"Gave up after {max_attempts} attempts. Last problem: {last}")

def query(params=None, flags=()):
    """Build a query string. Some NVD parameters (noRejected, hasKev) are bare flags with no
    value, which a params dict cannot express, so they are appended by hand."""
    parts = [urlencode(params or {})] + list(flags)
    return "&".join(p for p in parts if p)

print("helper ready")

helper ready


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 05 &nbsp;&middot;&nbsp; ACCESS CHECK</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Is the NVD API reachable?</div>
</div>


**Why** — the cheapest possible question, asked before anything expensive. One record, nothing parsed.

**Watch where the key goes** — `{"apiKey": ...}` is a **header**, never a query parameter. Put it in
the URL and NVD ignores it silently, dropping you to the anonymous rate limit.

**Pass** — `version 2.0` and `totalResults` near **400,000**.

| If you see | It probably means |
|:--|:--|
| A much smaller `totalResults` | An unintended filter on the query |
| "Cannot reach ..." | Firewall, VPN, or captive network |
| A TLS error | Something is intercepting HTTPS — try off the university VPN |

In [5]:
NVD_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
NVD_HEADERS = {"apiKey": NVD_API_KEY}      # a header, never a query parameter
try:
    payload = http_get(NVD_URL, params={"resultsPerPage": 1}, headers=NVD_HEADERS).json()
    total = payload["totalResults"]
    print(f"version {payload['version']}  |  timestamp {payload['timestamp']}")
    print(f"totalResults: {total:,}")
    record("05-nvd-reachable", total > 300_000, total_results=total,
           nvd_timestamp=payload["timestamp"])
except (ApiError, ValueError, KeyError) as exc:          # ValueError: a body that isn't JSON (A6)
    record("05-nvd-reachable", False, error=repr(exc))
    print("\n" + str(exc))

version 2.0  |  timestamp 2026-11-10T08:19:44.222
totalResults: 421,911
PASS  05-nvd-reachable  |  {'total_results': 421911, 'nvd_timestamp': '2026-11-10T08:19:44.222'}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 06 &nbsp;&middot;&nbsp; AUTHORISATION</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Is the key actually being honoured?</div>
</div>


**Why this needs its own test** — NVD never says *"your key is invalid."* If the key is wrong,
misplaced, or never activated, you are simply treated as anonymous:

| | Requests allowed (NVD's documented limits) |
|:--|:--|
| With a working key | **50 per rolling 30 seconds** |
| Anonymous | **5 per rolling 30 seconds** |

**How** — fire 10 requests 0.6 s apart (30 s ÷ 50 = 0.6 s), far above the anonymous allowance.

**Pass** — ten `200`s. If not, the cell names the likely cause from the status codes it actually saw:
a 403 or 429 points at the key; a 5xx means NVD itself is having trouble.
Step 06b checks that this test *can* fail.

<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">OBSERVED ON OCTOBER 9</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">Ten <code>200</code>s, as expected &mdash; but step 06b showed that ten <i>anonymous</i> requests at the same pace also succeed. So this step proves NVD is answering, not that the key is applied. That's acceptable: the full download pauses 6 s between requests, which stays within even the anonymous limit (5 per 30 s), so the project doesn't depend on the key being honoured.</div>
</div>

In [6]:
statuses = []
t0 = time.perf_counter()                 # an interval clock: immune to system-clock changes (A14)
try:
    for _ in range(10):
        resp = SESSION.get(NVD_URL, params={"resultsPerPage": 1},
                           headers=NVD_HEADERS, timeout=30)
        statuses.append(resp.status_code)
        time.sleep(0.6)                   # 30 s / 50 requests: the keyed limit's average pace (A15)
    elapsed = time.perf_counter() - t0
    counts = dict(Counter(statuses))
    all_ok = all(s == 200 for s in statuses)
    record("06-key-honoured", all_ok, statuses=statuses, counts=counts, seconds=round(elapsed, 1))
    if not all_ok:                        # a hint that matches what was actually seen (A13)
        if 403 in counts or 429 in counts:
            print("\n  Throttled, so the key is probably not being applied. Check that the header")
            print("  name is exactly 'apiKey', and that you clicked the activation link NIST emailed.")
        elif any(s >= 500 for s in counts):
            print("\n  Server errors (5xx): NVD itself is struggling or in maintenance.")
            print("  That says nothing about your key - re-run this cell later.")
        else:
            print(f"\n  Unexpected statuses {counts} - read them before blaming the key.")
except requests.RequestException as exc:
    record("06-key-honoured", False, error=repr(exc))

PASS  06-key-honoured  |  {'statuses': [200, 200, 200, 200, 200, 200, 200, 200, 200, 200], 'counts': {200: 10}, 'seconds': 8.0}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 06b &nbsp;&middot;&nbsp; NEGATIVE CONTROL</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">The same burst, without the key</div>
</div>


**Why** — a test that has never failed can't prove it would. Step 06 only means something if
anonymous requests really are refused at that pace. This is the first time we *observe* that,
rather than take it from documentation.

**Pass** — at least one request refused with a 4xx code. The code NVD uses for throttling is
recorded (it answers an open question, V18). The cell then waits 30 s so the anonymous burst
can't affect the steps after it.

<div style="background:#e5e2ee;border-left:4px solid #8a3324;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#8a3324;">OBSERVED ON OCTOBER 9 &mdash; WHY THIS STEP SHOWS FAIL</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">All ten anonymous requests returned <code>200</code>: NVD did not enforce its documented anonymous limit at this pace. The FAIL is the honest verdict &mdash; the control could not demonstrate that step 06 can fail &mdash; and it is a finding, not a bug. The status code NVD uses for throttling remains unobserved.</div>
</div>

In [7]:
# The same burst WITHOUT the key. If step 06 means anything, this one should be refused (A12).
anon_statuses, anon_reasons = [], set()
try:
    for _ in range(10):
        resp = SESSION.get(NVD_URL, params={"resultsPerPage": 1}, timeout=30)
        anon_statuses.append(resp.status_code)
        if resp.status_code != 200:
            anon_reasons.add(reason(resp)[:120])
        time.sleep(0.6)
    refused = sorted({s for s in anon_statuses if s != 200})
    print("anonymous statuses:", anon_statuses)
    print("reasons given     :", anon_reasons or "none")
    REPORT["findings"]["rate_limit_status_codes"] = refused            # answers V18
    # PASS means step 06 CAN fail: without a key, NVD refuses with a 4xx code.
    record("06b-negative-control", any(400 <= s < 500 for s in refused),
           statuses=anon_statuses, refused_with=refused)
    if not refused:
        print("\n  No refusals without a key, so step 06 proves nothing either way.")
except requests.RequestException as exc:
    record("06b-negative-control", False, error=repr(exc))
print("\nwaiting 30 s so the anonymous burst can't spill into the steps that follow ...")
time.sleep(30)

anonymous statuses: [200, 200, 200, 200, 200, 200, 200, 200, 200, 200]
reasons given     : none
FAIL  06b-negative-control  |  {'statuses': [200, 200, 200, 200, 200, 200, 200, 200, 200, 200], 'refused_with': []}

  No refusals without a key, so step 06 proves nothing either way.

waiting 30 s so the anonymous burst can't spill into the steps that follow ...


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 07 &nbsp;&middot;&nbsp; SEMANTICS</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">The known-answer probe: Log4Shell</div>
</div>


**Why** — previous steps proved bytes arrive. This one proves the bytes **mean what we think**.

**Why this record** — `CVE-2021-44228` is the best-documented vulnerability of the decade, so some
facts are known in advance. The code now **checks** them: the ID matches, it was published in 2021,
the description mentions Log4j, and a CVSS v3.1 base score of 10.0 exists. (Step 12 adds one more:
it is in KEV.)

**Read the output for these things**

| Look at | Why it matters |
|:--|:--|
| **Who supplied each CVSS score** | Each entry is tagged CNA, NVD or CISA-ADP. Only the CNA's existed at publication. |
| **Who supplied each CWE** | Same question for weakness classes — and NVD's `type` label (Primary / Secondary) next to it |
| **The CNA** | `security@apache.org` — the organisation that published the record |
| **References** | Many — but references accrue *after* publication, so they are never features |
| **Timestamp format** | Milliseconds, no timezone marker — this dictates how we parse dates in step 10 |

In [8]:
ADP_CISA = "134c704f-9b21-4f2e-91b3-4a467353bcc0"   # CISA's Authorized Data Publisher - not a CNA

def source_kind(source, cna):
    """Who produced a score or CWE entry: the CVE's own CNA, NVD's analysts, CISA-ADP, or other."""
    if not source:
        return None
    if source == cna:
        return "CNA"
    if source == "nvd@nist.gov":
        return "NVD"
    if source == ADP_CISA:
        return "CISA-ADP"
    return "other"

def english(cve):
    """The English description (a record can carry several languages)."""
    return next((d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"), "")

try:
    cve = http_get(NVD_URL, params={"cveId": "CVE-2021-44228"},
                   headers=NVD_HEADERS).json()["vulnerabilities"][0]["cve"]
    cna = cve.get("sourceIdentifier")
    print("id         :", cve["id"])
    print("published  :", cve["published"])
    print("vulnStatus :", cve.get("vulnStatus"))
    print("CNA        :", cna)
    print("description:", english(cve)[:140], "...")       # English, as flatten() uses (A17)
    print("references :", len(cve.get("references", [])))

    # Every entry WITH its producer and type: the provenance the A1 fix depends on (A18).
    print("\nCWE entries         who        type        source")
    for w in cve.get("weaknesses", []):
        for d in w.get("description", []):
            print(f"  {d.get('value', ''):<17} {str(source_kind(w.get('source'), cna)):<10} "
                  f"{str(w.get('type')):<11} {w.get('source')}")
    print("\nCVSS entries        score  who        type        source")
    v31_scores = []
    for key, entries in cve.get("metrics", {}).items():
        for m in entries:
            score = m.get("cvssData", {}).get("baseScore")
            if key == "cvssMetricV31":
                v31_scores.append(score)
            print(f"  {key:<17} {str(score):<6} {str(source_kind(m.get('source'), cna)):<10} "
                  f"{str(m.get('type')):<11} {m.get('source')}")

    # Known answers, checked by code rather than by eye (A19).
    print()
    failed = show_checks({
        "id matches the request": cve["id"] == "CVE-2021-44228",
        "published in 2021": str(cve["published"]).startswith("2021-"),
        "description mentions Log4j": "log4j" in english(cve).lower(),
        "a CVSS v3.1 base score of 10.0 exists": 10.0 in v31_scores,
    })
    record("07-known-record", not failed, failed=failed)
except (ApiError, ValueError, KeyError, IndexError) as exc:
    record("07-known-record", False, error=repr(exc))

id         : CVE-2021-44228
published  : 2021-12-10T10:15:09.143
vulnStatus : Analyzed
CNA        : security@apache.org
description: Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log mess ...
references : 103

CWE entries         who        type        source
  CWE-20            CNA        Secondary   security@apache.org
  CWE-400           CNA        Secondary   security@apache.org
  CWE-502           CNA        Secondary   security@apache.org
  CWE-917           NVD        Secondary   nvd@nist.gov

CVSS entries        score  who        type        source
  cvssMetricV31     10.0   NVD        Primary     nvd@nist.gov
  cvssMetricV31     10.0   CISA-ADP   Secondary   134c704f-9b21-4f2e-91b3-4a467353bcc0
  cvssMetricV2      9.3    NVD        Primary     nvd@nist.gov
  ssvcV203          None   CISA-ADP   None        134c704f-9b21-4f2e-91b3-4a467353bcc0

  ok  id matches the request
  ok  published in 20

<div style="background:#e5e2ee;border-left:4px solid #8a3324;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#8a3324;">WHY THIS MATTERS FOR THE MODEL</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">A score or CWE can come from three producers. The <b>CNA</b> supplies it at publication. <code>nvd@nist.gov</code> (a government analyst) and <b>CISA-ADP</b> (<code>134c704f&hellip;</code>, CISA's Authorized Data Publisher) add theirs <i>later</i> &mdash; and since NVD's April 2026 triage change, analysts reach <i>already-exploited</i> CVEs first. So model features use <b>CNA-supplied values only</b>; a CVE with no CNA score gets an explicit &ldquo;severity missing&rdquo; flag instead of someone else's score.</div>
</div>

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 08 &nbsp;&middot;&nbsp; CONSTRAINTS</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Break the rules on purpose</div>
</div>


**Why** — learn exactly how the API refuses an illegal request, while it costs nothing. Discovering a
limit in the middle of the full download would mean restarting a long job.

**How** — ask for a full calendar year (366 days — 2024 is a leap year), knowing the cap is **120 days**.

**Pass** — status **404** *and* a `message` header that mentions the 120-day limit. Any other
refusal (a 403 throttle, a 503 outage) is not proof that we understand the limit, so it fails.

**Worth absorbing** — the reason arrives in a custom HTTP header named `message`, with an **empty
body**. `http_get` (step 04) now reads that header when it reports an error.

In [9]:
try:
    bad = SESSION.get(NVD_URL, params={
        "pubStartDate": "2024-01-01T00:00:00.000Z",
        "pubEndDate":   "2024-12-31T23:59:59.999Z",   # a full calendar year: 366 days, 2024 is a leap year (A21)
        "resultsPerPage": 1,
    }, headers=NVD_HEADERS, timeout=30)
    message = bad.headers.get("message", "")
    print("status        :", bad.status_code)
    print("message header:", message or "(none)")
    print("body          :", bad.text[:200] or "(empty)")
    # The expected refusal is a 404 whose message names the 120-day cap. A 403 (throttled) or
    # 503 (NVD down) is also "not 200", but would prove nothing about the limit (A20).
    understood = bad.status_code == 404 and "120" in message
    record("08-date-limit-understood", understood, status=bad.status_code, message=message)
except requests.RequestException as exc:
    record("08-date-limit-understood", False, error=repr(exc))

status        : 404
message header: Date range cannot exceed 120 days.
body          : (empty)
PASS  08-date-limit-understood  |  {'status': 404, 'message': 'Date range cannot exceed 120 days.'}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 09 &nbsp;&middot;&nbsp; ACQUISITION</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">The reusable client, and proof its arithmetic is right</div>
</div>


**Why** — package everything learned so far into one object, and verify the window arithmetic
*before* trusting it with a long job.

| Component | What it does |
|:--|:--|
| `iter_cves()` | A **generator**: fetches a page of up to 2,000 records, yields them one at a time, moves on. Only one page of raw JSON is held at a time. At the end it **reconciles**: records received vs NVD's `totalResults`. |
| `count()` | Asks only for the total — one tiny request instead of a download |
| `windows()` | Yields date pairs at most 119 days wide — one day inside the cap, deliberately |
| `check_windows()` | Checks width, **contiguity** (no gaps or overlaps) and **coverage**, in exact time |
| A 6-second pause | NIST asks bulk users to sleep several seconds between requests |

**Pass** — `28 windows for 2016-2024`, no problems, and a deliberately broken window list is caught.

In [10]:
class NVDClient:
    """NVD CVE API 2.0: rate limited, retrying, paginating, self-reconciling."""
    URL = NVD_URL                       # one definition, shared with the steps above
    PAGE = 2000
    MAX_WINDOW_DAYS = 120
    FMT = "%Y-%m-%dT%H:%M:%S.000Z"

    def __init__(self, api_key, pause=6.0, timeout=120):
        # NIST asks bulk users to sleep several seconds between requests (A16).
        self.headers = {"apiKey": api_key}
        self.pause = pause
        self.timeout = timeout
        self.last_run = {}

    def get(self, params=None, flags=()):
        """One request; returns the parsed JSON page."""
        return http_get(self.URL, params=query(params, flags), headers=self.headers,
                        pause=self.pause, timeout=self.timeout).json()

    def count(self, flags=(), **params):
        """How many CVEs match a query, without downloading them: one tiny request."""
        return self.get({**params, "resultsPerPage": 1}, flags)["totalResults"]

    def iter_cves(self, progress=True, flags=(), **params):
        """Yield every CVE dict matching a query, page by page, then reconcile the count (A23)."""
        start, expected, received = 0, None, 0
        while expected is None or start < expected:
            page = self.get({**params, "resultsPerPage": self.PAGE, "startIndex": start}, flags)
            total = page["totalResults"]
            if expected is None:
                expected = total
            elif total != expected:
                print(f"  WARNING: totalResults changed mid-query: {expected:,} -> {total:,}")
            batch = page["vulnerabilities"]
            if not batch:
                print(f"  WARNING: empty page at startIndex {start:,} of {expected:,}")
                break
            for item in batch:
                yield item["cve"]
            received += len(batch)
            start += len(batch)
            if progress:
                print(f"  {start:,}/{expected:,}")
        self.last_run = {"expected": expected, "received": received,
                         "complete": received == expected}
        if received != expected:
            print(f"  WARNING: received {received:,} of {expected:,} records")

    def windows_between(self, start, stop):
        """Yield (start, end) ISO pairs covering [start, stop), each at most 119 days wide."""
        step = dt.timedelta(days=self.MAX_WINDOW_DAYS - 1)
        cursor = start
        while cursor < stop:
            end = min(cursor + step, stop)
            yield cursor.strftime(self.FMT), end.strftime(self.FMT)
            cursor = end

    def windows(self, start_year, end_year):
        """Whole calendar years: Jan 1 of start_year up to Jan 1 after end_year."""
        return self.windows_between(dt.datetime(start_year, 1, 1), dt.datetime(end_year + 1, 1, 1))

def check_windows(pairs, start, stop, max_days=120):
    """Width, contiguity and coverage, measured in exact time rather than whole days (A24)."""
    parse = lambda s: dt.datetime.strptime(s, NVDClient.FMT)
    widest = max(parse(b) - parse(a) for a, b in pairs)
    problems = []
    if widest > dt.timedelta(days=max_days):
        problems.append(f"a window is wider than {max_days} days")
    breaks = sum(1 for i in range(1, len(pairs)) if pairs[i][0] != pairs[i - 1][1])
    if breaks:
        problems.append(f"{breaks} gap(s) or overlap(s) between windows")
    if parse(pairs[0][0]) != start or parse(pairs[-1][1]) != stop:
        problems.append("windows don't start and end exactly on the requested dates")
    return problems, widest

client = NVDClient(NVD_API_KEY)
span_start, span_stop = dt.datetime(2016, 1, 1), dt.datetime(2025, 1, 1)
windows = list(client.windows(2016, 2024))
problems, widest = check_windows(windows, span_start, span_stop)
print(f"{len(windows)} windows for 2016-2024, widest {widest}")
for p in problems:
    print("  problem:", p)

# Prove the check can fail: shift one window's start by a day, leaving a gap. It must be caught.
shifted = dt.datetime.strptime(windows[5][0], NVDClient.FMT) + dt.timedelta(days=1)
broken = windows[:5] + [(shifted.strftime(NVDClient.FMT), windows[5][1])] + windows[6:]
gap_caught = bool(check_windows(broken, span_start, span_stop)[0])
print("a deliberate 1-day gap is caught:", gap_caught)
record("09-client-built", not problems and gap_caught,
       windows=len(windows), widest=str(widest), gap_caught=gap_caught)

28 windows for 2016-2024, widest 119 days, 0:00:00
a deliberate 1-day gap is caught: True
PASS  09-client-built  |  {'windows': 28, 'widest': '119 days, 0:00:00', 'gap_caught': True}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 10 &nbsp;&middot;&nbsp; PARSING</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Nested JSON to flat rows</div>
</div>


**Why** — one API record is deeply nested; we need one flat row.

**The rule that decides the features** — severity (CVSS) and weakness (CWE) come **only from the
CVE's own CNA**, found by matching each entry's `source` to the record's `sourceIdentifier`.
Every CVSS version is searched before giving up, so a CNA that scored only in v4.0 still counts.
If the CNA gave no score, `cvss_missing` is `True` — the score is *not* borrowed from NVD or CISA-ADP.

| Column group | Use |
|:--|:--|
| `cve_id`, `published`, `cna`, `description`, `cvss_*`, `cwe*` | Candidate features (publication-time) |
| `rejected` | A filter: withdrawn records are dropped in Stage 2 |
| `diag_*` | **Diagnostics only** — added after publication or by NVD's workflow. Never model features. |

**Also here** — `calendar_days()`, so a same-day KEV listing counts as 0 days, not −1.

**Pass** — ten self-tests on hand-made records with known right answers all print `ok`.

In [11]:
def to_naive_utc(series):
    """Parse ISO timestamps (with or without a trailing Z) to naive UTC datetimes.
    NVD returns naive UTC strings and KEV returns plain dates; normalising both the same way
    keeps later subtraction from failing on a timezone mismatch. Unparseable values become
    NaT, so callers must check for them (A29)."""
    return pd.to_datetime(series, errors="coerce", format="ISO8601",
                          utc=True).dt.tz_localize(None)

def calendar_days(later, earlier):
    """Whole calendar days between two date columns: a same-day pair is 0, never -1 (A2)."""
    return (later.dt.normalize() - earlier.dt.normalize()).dt.days

CVSS_VERSIONS = [("cvssMetricV31", "3.1"), ("cvssMetricV30", "3.0"),
                 ("cvssMetricV40", "4.0"), ("cvssMetricV2", "2.0")]

def flatten(cve):
    """One CVE dict -> one flat row.

    Model features may use only what the CVE's own CNA supplied (B1). Scores and CWEs from
    NVD analysts or CISA-ADP arrive after publication, so they are kept only in columns
    prefixed `diag_` - diagnostics that must never enter a model (blacklist: B2-B4, B14)."""
    cna = cve.get("sourceIdentifier")
    metrics = cve.get("metrics", {})

    def first_score(kind):
        # Search EVERY version, in priority order, before giving up (A26).
        for key, label in CVSS_VERSIONS:
            for m in metrics.get(key, []):
                if source_kind(m.get("source"), cna) == kind:
                    data = m.get("cvssData", {})
                    return (data.get("baseScore"), data.get("vectorString"),
                            data.get("baseSeverity") or m.get("baseSeverity"), label)
        return (None, None, None, None)

    score, vector, severity, version = first_score("CNA")            # A1
    nvd_score, _, _, nvd_version = first_score("NVD")
    adp_score, _, _, adp_version = first_score("CISA-ADP")
    score_sources = sorted({f"{label}:{source_kind(m.get('source'), cna)}"
                            for key, label in CVSS_VERSIONS for m in metrics.get(key, [])})

    # CWEs: the CNA's own list, in the CNA's order (A3, A27); NVD placeholders kept apart (A28).
    cwe_cna, cwe_other, placeholders, entry_types = [], [], [], []
    for w in cve.get("weaknesses", []):
        kind = source_kind(w.get("source"), cna)
        entry_types.append(f"{kind}:{w.get('type')}")
        for d in w.get("description", []):
            value = str(d.get("value", ""))
            if value.startswith("CWE-"):
                target = cwe_cna if kind == "CNA" else cwe_other
                if value not in target:
                    target.append(value)
            elif value:
                placeholders.append(f"{value}:{kind}")

    text = english(cve)
    refs = cve.get("references", [])
    return {
        "cve_id": cve.get("id"),
        "published": cve.get("published"),
        "rejected": cve.get("vulnStatus") == "Rejected",           # a filter, not a feature (B5)
        "cna": cna,
        "description": text,
        "description_len": len(text),
        "cvss_score": score,                                       # CNA only (B1)
        "cvss_severity": severity,
        "cvss_vector": vector,
        "cvss_version": version,
        "cvss_missing": score is None,                             # the B1 indicator
        "cwes": ";".join(cwe_cna),
        "cwe_primary": cwe_cna[0] if cwe_cna else None,
        "cwe_missing": not cwe_cna,
        # ---- diagnostics only: post-publication or NVD-workflow fields --------------------
        "diag_vuln_status": cve.get("vulnStatus"),                         # B4
        "diag_last_modified": cve.get("lastModified"),                     # B4
        "diag_nvd_score": nvd_score, "diag_nvd_version": nvd_version,
        "diag_adp_score": adp_score, "diag_adp_version": adp_version,
        "diag_score_sources": ";".join(score_sources),                     # V13 / V14
        "diag_cwes_non_cna": ";".join(cwe_other),                          # A3
        "diag_cwe_placeholders": ";".join(placeholders),                   # A28
        "diag_cwe_entry_types": ";".join(entry_types),                     # what "Primary" means
        "diag_n_references": len(refs),                                    # B3
        "diag_ref_tags": ";".join(sorted({t for r in refs for t in r.get("tags", [])})),  # B3
        "diag_n_configurations": len(cve.get("configurations", [])),       # B2
        "diag_has_nvd_kev_fields": "cisaExploitAdd" in cve,                # label-side (B14)
    }

# ---- Self-tests: hand-made records with known right answers ------------------------------
def _m(source, score, vector="AV:N"):
    return {"source": source, "type": "Secondary", "cvssData": {"baseScore": score,
            "vectorString": vector, "baseSeverity": "HIGH"}}

_cna = "psirt@vendor.example"
_rec = {
    "id": "CVE-0000-0001", "sourceIdentifier": _cna, "published": "2023-01-10T18:15:00.000",
    "vulnStatus": "Modified",
    "descriptions": [{"lang": "es", "value": "texto"}, {"lang": "en", "value": "English text"}],
    "metrics": {"cvssMetricV31": [_m("nvd@nist.gov", 9.8), _m(ADP_CISA, 9.1)],
                "cvssMetricV40": [_m(_cna, 8.7)]},
    "weaknesses": [{"source": _cna, "type": "Secondary",
                    "description": [{"lang": "en", "value": "CWE-79"}, {"lang": "en", "value": "CWE-1021"}]},
                   {"source": "nvd@nist.gov", "type": "Primary",
                    "description": [{"lang": "en", "value": "CWE-20"}]}],
}
_bare = {"id": "CVE-0000-0002", "sourceIdentifier": "cve@mitre.org", "vulnStatus": "Analyzed",
         "metrics": {"cvssMetricV31": [_m("nvd@nist.gov", 7.5)]},
         "weaknesses": [{"source": "nvd@nist.gov", "type": "Primary",
                         "description": [{"lang": "en", "value": "NVD-CWE-noinfo"}]}]}
r1, r2 = flatten(_rec), flatten(_bare)
_dates = to_naive_utc(pd.Series(["2023-01-10T18:15:00.000", "2023-01-10T18:15:00.000Z",
                                 "2023-01-10", "not a date"]))
failed = show_checks({
    "CNA's v4.0 score beats NVD/ADP v3.1 scores (A26)": (r1["cvss_score"], r1["cvss_version"]) == (8.7, "4.0"),
    "NVD and ADP scores kept only as diagnostics": (r1["diag_nvd_score"], r1["diag_adp_score"]) == (9.8, 9.1),
    "no CNA score -> missing flag, not an NVD score (B1)": r2["cvss_score"] is None and r2["cvss_missing"],
    "CWEs in the CNA's order, not alphabetical (A27)": r1["cwe_primary"] == "CWE-79",
    "NVD-added CWE kept out of the CNA list (A3)": r1["cwes"] == "CWE-79;CWE-1021" and r1["diag_cwes_non_cna"] == "CWE-20",
    "placeholder recorded, not silently dropped (A28)": r2["diag_cwe_placeholders"] == "NVD-CWE-noinfo:NVD",
    "English description picked (A17)": r1["description"] == "English text",
    "timestamps with and without Z agree": _dates[0] == _dates[1],
    "bad dates become NaT, to be counted (A29)": int(_dates.isna().sum()) == 1,
    "same-day listing is 0 days, not -1 (A2)": int(calendar_days(pd.Series([_dates[2]]), pd.Series([_dates[0]])).iloc[0]) == 0,
})
record("10-parser-selftest", not failed, tests=10, failed=failed)

  ok  CNA's v4.0 score beats NVD/ADP v3.1 scores (A26)
  ok  NVD and ADP scores kept only as diagnostics
  ok  no CNA score -> missing flag, not an NVD score (B1)
  ok  CWEs in the CNA's order, not alphabetical (A27)
  ok  NVD-added CWE kept out of the CNA list (A3)
  ok  placeholder recorded, not silently dropped (A28)
  ok  English description picked (A17)
  ok  timestamps with and without Z agree
  ok  bad dates become NaT, to be counted (A29)
  ok  same-day listing is 0 days, not -1 (A2)
PASS  10-parser-selftest  |  {'tests': 10, 'failed': []}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 11 &nbsp;&middot;&nbsp; FIRST REAL WORKLOAD</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Pull one 120-day window</div>
</div>


**Why** — one window proves paging, parsing, and error handling under real conditions. If this
works, the full download is the same code with a longer loop.

**How** — the first window of 2023, paged 2,000 records at a time, 6 s apart (about a minute).

**Pass** — all four conditions hold:

| Condition | What it guards against |
|:--|:--|
| Rows equal NVD's `totalResults` | Paging that silently stopped early |
| No unparseable publication dates | Rows silently vanishing from the time split |
| Every record published inside the window | Splitting on the wrong date (the year in a CVE ID is *not* the publication year) |
| No duplicate CVE ids | Double counting |

`Rejected` rows at the top are withdrawn CVE records. They are kept here and excluded from
every statistic below.

In [12]:
nvd = pd.DataFrame()
try:
    w_start, w_end = next(client.windows(2023, 2023))
    print(f"window {w_start[:10]} to {w_end[:10]}  (pausing {client.pause:.0f} s between pages)")
    rows = [flatten(c) for c in client.iter_cves(pubStartDate=w_start, pubEndDate=w_end)]
    note_source("NVD CVE API 2.0", query=f"pubStartDate={w_start}&pubEndDate={w_end}",
                records=len(rows))
    nvd = pd.DataFrame(rows)
    nvd["published"] = to_naive_utc(nvd["published"])
    nvd["diag_last_modified"] = to_naive_utc(nvd["diag_last_modified"])
    lo, hi = pd.Timestamp(w_start.rstrip("Z")), pd.Timestamp(w_end.rstrip("Z"))
    failed = show_checks({
        "rows equal NVD's totalResults (A23, A30)": client.last_run.get("complete", False),
        "no unparseable publication dates (A29)": int(nvd["published"].isna().sum()) == 0,
        "every record published inside the window (B11)": bool(nvd["published"].between(lo, hi).all()),
        "no duplicate CVE ids (A10)": not nvd["cve_id"].duplicated().any(),
    })
    record("11-nvd-window", not failed, rows=len(nvd), **client.last_run, failed=failed)
except (ApiError, ValueError, KeyError) as exc:
    nvd = pd.DataFrame()
    record("11-nvd-window", False, error=repr(exc))
print(nvd.shape)
nvd.head(3)

window 2023-01-01 to 2023-04-30  (pausing 6 s between pages)
  2,000/10,182
  4,000/10,182
  6,000/10,182
  8,000/10,182
  10,000/10,182
  10,182/10,182
  ok  rows equal NVD's totalResults (A23, A30)
  ok  no unparseable publication dates (A29)
  ok  every record published inside the window (B11)
  ok  no duplicate CVE ids (A10)
PASS  11-nvd-window  |  {'rows': 10182, 'expected': 10182, 'received': 10182, 'complete': True, 'failed': []}
(10182, 28)


,cve_id,published,rejected,cna,description,description_len,cvss_score,cvss_severity,cvss_vector,cvss_version,...,diag_adp_score,diag_adp_version,diag_score_sources,diag_cwes_non_cna,diag_cwe_placeholders,diag_cwe_entry_types,diag_n_references,diag_ref_tags,diag_n_configurations,diag_has_nvd_kev_fields
0,CVE-2022-36816,2023-01-01 01:15:10.057,True,security@atlassian.com,Rejected reason: To maintain compliance with C...,118,NaN,NaN,NaN,NaN,...,NaN,NaN,,,,,0,,0,False
1,CVE-2022-36827,2023-01-01 01:15:10.167,True,security@atlassian.com,Rejected reason: To maintain compliance with C...,118,NaN,NaN,NaN,NaN,...,NaN,NaN,,,,,0,,0,False
2,CVE-2022-43783,2023-01-01 01:15:10.223,True,security@atlassian.com,Rejected reason: To maintain compliance with C...,118,NaN,NaN,NaN,NaN,...,NaN,NaN,,,,,0,,0,False


<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">HOW TO READ THE PROVENANCE TABLES</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">All figures are over <b>live</b> records. <b>CNA score missing</b> is the share that will carry the &ldquo;severity missing&rdquo; flag. <b>Records with a score from &hellip;</b> shows how often NVD and CISA-ADP added their own. The <b>who &times; type</b> table answers what NVD's &ldquo;Primary&rdquo; label really marks. If more than 5% of records carry scores from an unrecognised source, the CNA-matching rule needs a look before anything is built on it.</div>
</div>

In [13]:
def pct(x):
    return f"{x:.1%}"

if nvd.empty:
    print("no rows: fix step 11 before continuing")
else:
    live = nvd[~nvd["rejected"]]
    print(f"records : {len(nvd):,}  =  {len(live):,} live  +  {int(nvd['rejected'].sum()):,} Rejected")
    print("everything below is over LIVE records only (A31)\n")
    print("duplicate ids        :", int(nvd["cve_id"].duplicated().sum()))
    print("CNA score missing    :", pct(live["cvss_missing"].mean()), "  <- becomes the B1 'severity missing' flag")
    print("CNA CWE missing      :", pct(live["cwe_missing"].mean()))

    print("\nCNA score, by CVSS version (B10):")
    print(live["cvss_version"].value_counts(dropna=False).to_string())

    # Who supplied ANY score - a record can carry several (V13 / V14).
    print("\nrecords with at least one score from ...")
    has_kind = {k: int(live["diag_score_sources"].str.contains(f":{k}", regex=False).sum())
                for k in ["CNA", "NVD", "CISA-ADP", "other"]}
    for k, n in has_kind.items():
        print(f"  {k:<9} {n:>6,}  ({n / len(live):.1%})")

    # A26 in practice: CNA scored only in a non-3.1 version while someone else supplied a v3.1.
    other_v31 = live["diag_score_sources"].str.contains(r"3\.1:(?:NVD|CISA-ADP|other)", regex=True)
    a26 = int((live["cvss_version"].notna() & (live["cvss_version"] != "3.1") & other_v31).sum())
    print(f"\nCNA score found only outside v3.1, while NVD/ADP had a v3.1 (A26 cases): {a26:,}")

    # What does NVD's weakness 'type' mean? Cross 'who' with 'type' (V13, A27).
    et = live["diag_cwe_entry_types"].str.split(";").explode().reset_index(drop=True)
    et = et[et.fillna("") != ""].str.split(":", n=1, expand=True)
    et.columns = ["who", "type"]
    type_by_kind = pd.crosstab(et["who"], et["type"])
    print("\nCWE entries: who supplied them  x  NVD's 'type' label")
    print(type_by_kind.to_string())

    cwe_src = np.select([live["cwes"] != "", live["diag_cwes_non_cna"] != ""],
                        ["CNA CWE", "only NVD/ADP CWE"], "no CWE at all")
    print("\nCWE coverage:")
    print(pd.Series(cwe_src).value_counts().to_string())

    other_share = has_kind["other"] / len(live)
    if other_share > 0.05:
        print(f"\n  CHECK: {other_share:.1%} of records carry scores from an unrecognised source -")
        print("  the CNA-matching rule may be missing a naming variant. Inspect before trusting B1.")

    print("\ntop CNAs, with the share of their records that carry a CNA score:")
    top = (live.groupby("cna").agg(records=("cve_id", "size"),
                                   cna_scored=("cvss_missing", lambda s: 1 - s.mean()))
               .sort_values("records", ascending=False).head(10))
    print(top.to_string(formatters={"cna_scored": pct}))

    REPORT["findings"]["window_provenance"] = {
        "records": len(nvd), "live": len(live), "rejected": int(nvd["rejected"].sum()),
        "cna_score_share": float(1 - live["cvss_missing"].mean()),
        "cna_cwe_share": float(1 - live["cwe_missing"].mean()),
        "cna_score_versions": {str(k): int(v) for k, v in
                               live["cvss_version"].fillna("none").value_counts().items()},
        "records_with_score_from": has_kind,
        "a26_cases": a26,
        "cwe_type_by_kind": {f"{k}|{t}": int(v) for (k, t), v in type_by_kind.stack().items()},
        "cwe_coverage": {k: int(v) for k, v in pd.Series(cwe_src).value_counts().items()},
        "placeholder_rows": int((live["diag_cwe_placeholders"] != "").sum()),
    }

records : 10,182  =  9,302 live  +  880 Rejected
everything below is over LIVE records only (A31)

duplicate ids        : 0
CNA score missing    : 40.9%   <- becomes the B1 'severity missing' flag
CNA CWE missing      : 44.2%

CNA score, by CVSS version (B10):
cvss_version
3.1    4820
NaN    3803
3.0     679

records with at least one score from ...
  CNA        5,499  (59.1%)
  NVD        8,598  (92.4%)
  CISA-ADP   3,307  (35.6%)
  other          0  (0.0%)

CNA score found only outside v3.1, while NVD/ADP had a v3.1 (A26 cases): 679

CWE entries: who supplied them  x  NVD's 'type' label
type      Primary  Secondary
who                         
CISA-ADP        0       2956
CNA             9       5179
NVD          6365        177

CWE coverage:
CNA CWE             5188
only NVD/ADP CWE    3374
no CWE at all        740

top CNAs, with the share of their records that carry a CNA score:
                                records cna_scored
cna                                               


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 12 &nbsp;&middot;&nbsp; LABELS</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">The CISA KEV catalog</div>
</div>


**Why** — this is the ground truth the whole project predicts: which CVEs are *confirmed* exploited
in the wild. No key, no rate limit, one JSON download.

**Pass** — the row count equals the catalog's own `count`, no duplicate IDs, every date parses,
and Log4Shell is listed.

**Read three things in the output**

| Output | What to take from it |
|:--|:--|
| `count` vs. 400,000 CVEs | The class imbalance: about **4 in every 1,000**. This dictates the entire evaluation design. |
| **Additions per year** and the **launch-day bulk load** | KEV began on Nov 3, 2021 with a few hundred entries at once — older CVEs were listed late by construction |
| The column list | Schemas change. Only `cveID` and `dateAdded` will ever join the analysis table. |

In [14]:
KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
kev = pd.DataFrame()
try:
    kev_raw = http_get(KEV_URL).json()
    note_source("CISA KEV catalog", catalog_version=kev_raw.get("catalogVersion"),
                date_released=kev_raw.get("dateReleased"), count=kev_raw.get("count"))
    print("catalog :", kev_raw.get("title"))
    print("version :", kev_raw.get("catalogVersion"))
    print("released:", kev_raw.get("dateReleased"))
    print("count   :", kev_raw.get("count"))
    kev = pd.DataFrame(kev_raw["vulnerabilities"])
    kev["cveID"] = kev["cveID"].str.strip().str.upper()        # join keys normalised (A36)
    kev["dateAdded"] = to_naive_utc(kev["dateAdded"])
    print("\ncolumns:", list(kev.columns))
    print("\nadditions per year:")
    print(kev["dateAdded"].dt.year.value_counts().sort_index().to_string())
    first_day = kev["dateAdded"].min()
    n_first = int((kev["dateAdded"] == first_day).sum())
    print(f"\nlaunch-day bulk load: {n_first} entries on {first_day:%Y-%m-%d} (see B13)")
    print()
    failed = show_checks({
        "rows equal the catalog's own count (A32)": len(kev) == kev_raw.get("count"),
        "no duplicate CVE ids": not kev["cveID"].duplicated().any(),
        "no unparseable dateAdded (A29)": int(kev["dateAdded"].isna().sum()) == 0,
        "Log4Shell is listed - a known answer (A19)": "CVE-2021-44228" in set(kev["cveID"]),
    })
    record("12-kev", not failed, entries=len(kev), launch_day_entries=n_first, failed=failed)
except (ApiError, ValueError, KeyError) as exc:
    kev = pd.DataFrame()
    record("12-kev", False, error=repr(exc))
kev.head(3)

catalog : CISA Catalog of Known Exploited Vulnerabilities
version : 2026.11.09
released: 2026-11-09T19:54:37.4945Z
count   : 1773

columns: ['cveID', 'vendorProject', 'product', 'vulnerabilityName', 'dateAdded', 'shortDescription', 'requiredAction', 'dueDate', 'knownRansomwareCampaignUse', 'forensicTriage', 'notes', 'cwes']

additions per year:
dateAdded
2021    311
2022    555
2023    187
2024    186
2025    245
2026    289

launch-day bulk load: 287 entries on 2021-11-03 (see B13)

  ok  rows equal the catalog's own count (A32)
  ok  no duplicate CVE ids
  ok  no unparseable dateAdded (A29)
  ok  Log4Shell is listed - a known answer (A19)
PASS  12-kev  |  {'entries': 1773, 'launch_day_entries': 287, 'failed': []}


,cveID,vendorProject,product,vulnerabilityName,dateAdded,shortDescription,requiredAction,dueDate,knownRansomwareCampaignUse,forensicTriage,notes,cwes
0,CVE-2026-102895,GPUStack,GPUStack,GPUStack Improper Authentication Vulnerability,2026-11-09,GPUStack contains an improper authentication v...,Apply mitigations in accordance with vendor in...,2026-11-12,Unknown,Yes,This vulnerability affects an open-source comp...,"[CWE-287, CWE-290, CWE-346]"
1,CVE-2026-123158,OpenBSD,OpenSSH,OpenBSD OpenSSH Unexpected Status Code or Retu...,2026-11-09,OpenBSD OpenSSH on Linux contains an unexpecte...,Apply mitigations in accordance with vendor in...,2026-11-12,Unknown,Yes,This vulnerability affects an open-source comp...,[CWE-394]
2,CVE-2026-116779,SignalWire,FreeSwitch,SignalWire FreeSwitch Code Injection Vulnerabi...,2026-11-09,SignalWire FreeSwitch contains a code injectio...,Apply mitigations in accordance with vendor in...,2026-11-12,Unknown,Yes,This vulnerability affects an open-source comp...,"[CWE-77, CWE-78, CWE-749]"


<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">HOW TO READ THE ADDITIONS-PER-YEAR TABLE</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;"><code>dateAdded</code> is when CISA <i>listed</i> the CVE &mdash; confirmation, not exploitation. For CVEs published before the catalog existed, it mostly records the 2021 launch and the 2022 catch-up. That is why the model uses only CVEs published from Nov 3, 2021 onward (step 17 checks there are enough of them).</div>
</div>

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 13 &nbsp;&middot;&nbsp; BENCHMARK</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">EPSS — the industry-standard score</div>
</div>


**Why** — EPSS is what our model will be compared against. This step confirms it exists, scores
every live CVE in the window, and records how the API behaves.

**What EPSS means** — the probability of exploitation activity **in the next 30 days**. Our label is
different: listed in KEV **within 365 days of publication**. The comparison is still useful, but the
two answer different questions.

**How** — 100 CVEs per request, with the page size set explicitly. One extra request without a
page size records the API's default (an open question, V22).

**Pass** — at least 98% of live CVEs scored, no duplicates, nothing unrequested. EPSS never scores
withdrawn records, so they are left out of the lookup.

In [15]:
EPSS_URL = "https://api.first.org/data/v1/epss"
EPSS_CHUNK = 100
epss = pd.DataFrame(columns=["cve", "epss", "percentile", "date"])

def fetch_epss(cve_ids, chunk=EPSS_CHUNK, **extra):
    """Yield EPSS records for CVE ids, `chunk` per request. The page size is set explicitly,
    so a bigger chunk can never be cut short by the API's default page size (A34)."""
    ids = list(cve_ids)
    for i in range(0, len(ids), chunk):
        part = ids[i:i + chunk]
        body = http_get(EPSS_URL, params={"cve": ",".join(part), "limit": len(part), **extra},
                        pause=0.5).json()
        yield from body.get("data", [])

try:
    meta = http_get(EPSS_URL, params={"limit": 1}).json()
    score_date = meta["data"][0]["date"] if meta.get("data") else None
    print(f"EPSS scores {meta['total']:,} CVEs  |  score date {score_date}")
    print("response fields:", sorted(meta))

    # V22: what page size does the API use when we don't say? (no limit given here on purpose)
    probe = http_get(EPSS_URL).json()
    default_limit = probe.get("limit")
    print(f"default page size (V22): limit={default_limit}, rows returned={len(probe.get('data', []))}")
    REPORT["findings"]["epss_default_limit"] = default_limit
    REPORT["findings"]["epss_response_fields"] = sorted(meta)

    if nvd.empty:
        record("13-epss", False, error="no NVD rows to look up")
    else:
        # All live records, not head(500): EPSS never scores Rejected records (V2, A33).
        wanted = nvd.loc[~nvd["rejected"], "cve_id"].tolist()
        print(f"\nlooking up all {len(wanted):,} live CVEs, {EPSS_CHUNK} per request ...")
        found = list(fetch_epss(wanted))
        note_source("EPSS (FIRST)", score_date=score_date, total_scored=meta["total"],
                    model_version=meta.get("model_version"))
        if found:
            epss = pd.DataFrame(found)
            epss["cve"] = epss["cve"].str.strip().str.upper()
            for col in ("epss", "percentile"):
                epss[col] = pd.to_numeric(epss[col], errors="coerce")
        coverage = len(epss) / len(wanted) if wanted else 0.0
        print(f"returned {len(epss):,} of {len(wanted):,} live CVEs ({coverage:.1%} coverage)")
        if len(epss):
            print(epss["epss"].describe().round(4).to_string())
        print()
        failed = show_checks({
            "at least 98% of live CVEs scored": coverage >= 0.98,
            "no duplicate ids returned": not epss["cve"].duplicated().any(),
            "nothing returned that wasn't asked for": set(epss["cve"]) <= set(wanted),
        })
        record("13-epss", not failed, total_scored=meta["total"], requested=len(wanted),
               returned=len(epss), coverage=round(coverage, 4), failed=failed)
except (ApiError, ValueError, KeyError) as exc:
    record("13-epss", False, error=repr(exc))
epss.head(3)

EPSS scores 402,199 CVEs  |  score date 2026-11-09
response fields: ['access', 'data', 'limit', 'offset', 'status', 'status-code', 'total', 'version']
default page size (V22): limit=100, rows returned=100

looking up all 9,302 live CVEs, 100 per request ...
returned 9,302 of 9,302 live CVEs (100.0% coverage)
count    9302.0000
mean        0.0243
std         0.1017
min         0.0006
25%         0.0038
50%         0.0061
75%         0.0094
max         1.0000

  ok  at least 98% of live CVEs scored
  ok  no duplicate ids returned
  ok  nothing returned that wasn't asked for
PASS  13-epss  |  {'total_scored': 402199, 'requested': 9302, 'returned': 9302, 'coverage': 1.0, 'failed': []}


,cve,epss,percentile,date
0,CVE-2023-22551,0.01589,0.75817,2026-11-09
1,CVE-2023-22452,0.00595,0.48732,2026-11-09
2,CVE-2023-22451,0.00740,0.54859,2026-11-09


<div style="background:#e5e2ee;border-left:4px solid #8a3324;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#8a3324;">TODAY'S SCORE IS NOT A FAIR BENCHMARK</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">Today's EPSS already reflects years of evidence, including the exploitation we want to predict. The fair benchmark is the score a CVE had <b>when it was published</b>. The next cell tries that rule on five exploited CVEs: the first score available on or after the publication date.</div>
</div>

In [16]:
# Historical EPSS (B6, B15): the score a CVE had right after publication, versus today.
# Rule under test: "first score available on or after the publication date", searching up to
# three days. Five exploited CVEs from the window are enough to see whether this works.
hist_rows = []
try:
    if nvd.empty or kev.empty:
        raise ApiError("needs steps 11 and 12")
    sample = (nvd[~nvd["rejected"] & nvd["cve_id"].isin(set(kev["cveID"]))]
              .sort_values("published").head(5))
    for cve_id, published in zip(sample["cve_id"], sample["published"]):
        first = None
        for offset in range(0, 3):
            day = (published + pd.Timedelta(days=offset)).strftime("%Y-%m-%d")
            data = http_get(EPSS_URL, params={"cve": cve_id, "date": day}, pause=0.5).json().get("data", [])
            if data:
                first = data[0]
                break
        today = epss.loc[epss["cve"] == cve_id, "epss"]
        hist_rows.append({"cve_id": cve_id, "published": f"{published:%Y-%m-%d}",
                          "first_score_date": first and first.get("date"),
                          "epss_then": first and float(first["epss"]),
                          "epss_today": float(today.iloc[0]) if len(today) else None,
                          "fields": first and sorted(first)})
    hist = pd.DataFrame(hist_rows)
    print(hist.drop(columns="fields").to_string(index=False))
    if len(hist) and hist["fields"].notna().any():
        print("\nfields in a historical record:", hist["fields"].dropna().iloc[0])
    REPORT["findings"]["epss_historical_probe"] = hist_rows
except (ApiError, ValueError, KeyError) as exc:
    print("historical EPSS probe failed:", repr(exc))
    REPORT["findings"]["epss_historical_probe"] = {"error": repr(exc)}

        cve_id  published first_score_date  epss_then  epss_today
CVE-2022-42475 2023-01-02       2023-01-03    0.01156     0.99474
CVE-2022-44877 2023-01-05       2023-01-06    0.00885     0.99995
CVE-2023-21674 2023-01-10       2023-01-11    0.00885     0.34206
CVE-2023-22952 2023-01-11       2023-01-12    0.00885     0.80139
CVE-2023-21839 2023-01-18       2023-01-18    0.00885     0.99900

fields in a historical record: ['cve', 'date', 'epss', 'percentile']


<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">READING THE COMPARISON</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">Expect <i>epss_then</i> to be far lower than <i>epss_today</i> for exploited CVEs. If historical scores come back empty, the benchmark rule needs adjusting before Stage 4.</div>
</div>

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 13b &nbsp;&middot;&nbsp; BENCHMARK DATA</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">EPSS's daily files, and which model was scoring</div>
</div>


**Why** — question Q4 compares our model with EPSS as it stood **1, 7 and 30 days after each 2024 CVE
was published**: about 120,000 historical scores. The API answers one date per request, so collecting
them that way would take thousands of requests. FIRST also publishes **one file per day containing every
score**, and about 400 of those cover the whole test year. This step proves the files are usable before
the plan depends on them.

**How** — four daily files are downloaded:

| File | Why this day |
|:--|:--|
| The probe date from step 13 | **Cross-check**: the file must agree with the API's historical answer, for the probe CVE and for 20 random CVEs |
| January 2, 2024 | The first scores the test year needs |
| January 31, 2025 | The last: 30 days after a CVE published on December 31, 2024 |
| The latest available | Which model is scoring **today** |

**Also recorded** — each file's first line names the **EPSS model version**. Version 3 of this notebook
recorded no model version (step 13's API response doesn't include one), so this fills that gap. The two
test-year files should name the same model, so the benchmark isn't split across two models.

**Pass** — every file downloads and parses, and file and API agree to five decimal places. A header
without a model version, a test year scored by two models, or a latest file whose size differs from
the API's total by more than 1% is a **WARN**.


In [17]:
# STEP 13b: EPSS daily files - where the full study's historical scores will come from (B6, B15).
import gzip, io

EPSS_FILE_HOSTS = ["https://epss.empiricalsecurity.com",   # current host, per FIRST's data page
                   "https://epss.cyentia.com"]             # the older host, tried only as a fallback
TEST_FIRST_DAY = "2024-01-02"      # day after the first 2024 publication day
TEST_LAST_DAY  = "2025-01-31"      # 30 days after a CVE published on Dec 31, 2024 (+1 day of slack)

def parse_epss_header(line):
    """'#model_version:v2025.03.14,score_date:2025-03-14T00:00:00+0000' -> dict.
    The layout isn't formally documented, so it is parsed loosely; the raw line is kept too."""
    fields = {}
    for part in (line or "").lstrip("#").split(","):
        key, sep, value = part.partition(":")
        if sep:
            fields[key.strip()] = value.strip()
    return fields

def epss_daily_file(day):
    """One day's complete EPSS file -> (scores, header dict, url, compressed size in bytes).
    Raises ApiError when no host serves that day."""
    name, problems = f"epss_scores-{day}.csv.gz", []
    for host in EPSS_FILE_HOSTS:
        try:
            resp = http_get(f"{host}/{name}", headers={"Accept": "*/*"},
                            max_attempts=2, pause=0.5, timeout=120)
        except ApiError as exc:
            problems.append(f"{host}: {exc}")
            continue
        raw = resp.content
        # A server may send the file pre-decompressed (Content-Encoding: gzip); check the magic bytes.
        text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8")
        first, _, rest = text.partition("\n")
        header = first.strip() if first.startswith("#") else ""
        scores = pd.read_csv(io.StringIO(rest if header else text))
        scores["cve"] = scores["cve"].astype(str).str.strip().str.upper()
        scores["epss"] = pd.to_numeric(scores["epss"], errors="coerce")
        return scores, {"raw": header, **parse_epss_header(header)}, f"{host}/{name}", len(raw)
    raise ApiError(f"no host served {name}: " + " | ".join(problems))

bulk_files, bulk_frames = {}, {}
try:
    probe_rows = [r for r in (REPORT["findings"].get("epss_historical_probe") or [])
                  if isinstance(r, dict) and r.get("first_score_date")]
    if not probe_rows:
        raise ApiError("needs a historical score from the probe above (step 13)")
    ref = probe_rows[0]

    # Today's file may not be published yet, so fall back a few days from step 13's score date.
    base = pd.Timestamp(REPORT["sources"].get("EPSS (FIRST)", {}).get("score_date") or dt.date.today())
    latest_candidates = [f"{base - pd.Timedelta(days=k):%Y-%m-%d}" for k in range(0, 4)]

    wanted_files = [("probe date", [ref["first_score_date"]]), ("test year starts", [TEST_FIRST_DAY]),
                    ("test horizon ends", [TEST_LAST_DAY]), ("latest", latest_candidates)]
    for label, days in wanted_files:
        for day in days:
            try:
                scores, header, url, size = epss_daily_file(day)
            except ApiError as exc:
                bulk_files[label] = {"day": day, "ok": False, "error": str(exc)[:200]}
                continue
            bulk_frames[label] = scores
            bulk_files[label] = {
                "day": day, "ok": True, "url": url, "rows": len(scores),
                "columns": list(scores.columns), "compressed_mb": round(size / 1e6, 2),
                "model_version": header.get("model_version"), "score_date": header.get("score_date"),
                "header": header["raw"][:120]}
            break
    note_source("EPSS daily files (FIRST)", files={k: v.get("url") for k, v in bulk_files.items()})

    print(f"{'file':<18} {'day':<11} {'rows':>9} {'MB':>6}  model version")
    for label, f in bulk_files.items():
        if f["ok"]:
            print(f"{label:<18} {f['day']:<11} {f['rows']:>9,} {f['compressed_mb']:>6}  {f['model_version']}")
        else:
            print(f"{label:<18} {f['day']:<11}  NOT DOWNLOADED: {f['error']}")

    # Cross-check 1: the probe CVE's score in the file equals the API's historical answer.
    probe_file = bulk_frames.get("probe date", pd.DataFrame(columns=["cve", "epss"]))
    in_file = probe_file.loc[probe_file["cve"] == ref["cve_id"], "epss"]
    probe_diff = abs(float(in_file.iloc[0]) - float(ref["epss_then"])) if len(in_file) else None
    # Cross-check 2: 20 random CVEs from the same file, asked of the API for the same date.
    api_diff, n_compared = None, 0
    if len(probe_file):
        ids = probe_file["cve"].sample(min(20, len(probe_file)), random_state=510).tolist()
        api = pd.DataFrame(list(fetch_epss(ids, date=bulk_files["probe date"]["day"])))
        if len(api):
            api["cve"] = api["cve"].str.strip().str.upper()
            api["epss"] = pd.to_numeric(api["epss"], errors="coerce")
            both = api.merge(probe_file[["cve", "epss"]], on="cve", suffixes=("_api", "_file"))
            n_compared = len(both)
            api_diff = float((both["epss_api"] - both["epss_file"]).abs().max()) if n_compared else None
    print(f"\nfile vs API on {ref['first_score_date']}: probe CVE {ref['cve_id']} differs by {probe_diff}; "
          f"largest difference over {n_compared} random CVEs: {api_diff}")

    versions = {label: f.get("model_version") for label, f in bulk_files.items() if f["ok"]}
    test_versions = {versions.get("test year starts"), versions.get("test horizon ends")} - {None}
    api_total = REPORT["steps"].get("13-epss", {}).get("total_scored")
    latest_rows = bulk_files.get("latest", {}).get("rows")
    rows_close = bool(api_total and latest_rows and abs(latest_rows - api_total) / api_total <= 0.01)
    n_days = (pd.Timestamp(TEST_LAST_DAY) - pd.Timestamp(TEST_FIRST_DAY)).days + 1
    sizes_mb = [f["compressed_mb"] for f in bulk_files.values() if f["ok"]]
    est_mb = round(n_days * float(np.mean(sizes_mb)), 0) if sizes_mb else None
    print(f"model versions: {versions}")
    print(f"Q4 needs about {n_days} daily files, roughly {est_mb} MB compressed "
          f"(only the test-set CVEs' rows need to be kept)")

    print()
    hard = show_checks({
        "every requested daily file downloaded and parsed": all(f["ok"] for f in bulk_files.values())
            and all({"cve", "epss", "percentile"} <= set(f["columns"]) for f in bulk_files.values()),
        "file agrees with the API for the probe CVE": probe_diff is not None and probe_diff < 1e-5,
        "file agrees with the API for 20 random CVEs": n_compared >= 18 and api_diff is not None and api_diff < 1e-5,
    })
    soft = show_checks({
        "every header names a model version": bool(versions) and all(versions.values()),
        "the test year was scored by a single model": len(test_versions) <= 1,   # unknown versions: see above
        "latest file's size within 1% of the API's total": rows_close,
    })
    REPORT["findings"]["epss_bulk"] = {
        "files": bulk_files, "model_versions": versions,
        "test_year_single_model": len(test_versions) == 1 if test_versions else None,
        "cross_check": {"probe_cve": ref["cve_id"], "probe_diff": probe_diff,
                        "random_compared": n_compared, "random_max_diff": api_diff},
        "q4_estimate": {"daily_files": n_days, "compressed_mb": est_mb}}
    record("13b-epss-bulk", (not hard) and (not soft or None),
           model_versions=versions, failed=hard + soft)
except (ApiError, ValueError, KeyError, OSError) as exc:
    record("13b-epss-bulk", False, error=repr(exc))


file               day              rows     MB  model version
probe date         2023-01-03    192,152   0.77  v2022.01.01
test year starts   2024-01-02    220,901    1.4  v2023.03.01
test horizon ends  2025-01-31    275,939   1.66  v2023.03.01
latest             2026-11-09    402,199   2.82  v2026.06.15

file vs API on 2023-01-03: probe CVE CVE-2022-42475 differs by 0.0; largest difference over 20 random CVEs: 0.0
model versions: {'probe date': 'v2022.01.01', 'test year starts': 'v2023.03.01', 'test horizon ends': 'v2023.03.01', 'latest': 'v2026.06.15'}
Q4 needs about 396 daily files, roughly 658.0 MB compressed (only the test-set CVEs' rows need to be kept)

  ok  every requested daily file downloaded and parsed
  ok  file agrees with the API for the probe CVE
  ok  file agrees with the API for 20 random CVEs
  ok  every header names a model version
  ok  the test year was scored by a single model
  ok  latest file's size within 1% of the API's total
PASS  13b-epss-bulk  |  {'model_

<div style="background:#e5e2ee;border-left:4px solid #8a3324;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#8a3324;">WHY THE MODEL VERSION MATTERS</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">EPSS is retrained from time to time, and a new version changes every score at once. If 2024 was scored by one model throughout, Q4 compares our model against <b>one</b> fixed benchmark. If the version changed mid-year, the comparison is reported separately for each version. Either way, the version is named in the report.</div>
</div>


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 14 &nbsp;&middot;&nbsp; INTEGRATION</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">The three-way join — the whole point</div>
</div>


**Why** — the project rests on an assumption that sounds trivial: that a CVE ID in NVD, in KEV, and
in EPSS is the same string and can be joined. If it were not, the project would be dead — and we
want to know that in week one, not week eight.

**What the join guarantees now**

- Only `dateAdded` comes across from KEV. Every other KEV column exists *only* for exploited CVEs,
  so it would hand the answer to a model. An assertion enforces this.
- A duplicate key anywhere stops the join with an error, instead of silently doubling rows.
- `days_to_kev` is in **calendar days**: 0 means listed the same day it was published; a negative
  value means listed *before* the publication day (rare — 4 of 50 in the October 1 sample).

**Pass** — a few dozen KEV hits (well under 2% of live rows) and no withdrawn record in KEV.
**Zero** hits is a WARN, not a FAIL: a genuinely quiet window is possible.

**Bonus output** — exploitation rates by CNA severity, and two simple rules (CNA CVSS ≥ 7;
CNA is Microsoft), recomputed on publication-time scores. These replace figures in the proposal.

In [18]:
merged = pd.DataFrame()

if nvd.empty or kev.empty:
    record("14-join", False, error="need both NVD rows and the KEV catalog")
else:
    try:
        # Only the label crosses over from KEV: every other KEV column exists only for
        # exploited CVEs, so it would hand the answer to a model (A35, B14).
        merged = (
            nvd
            .merge(kev[["cveID", "dateAdded"]], left_on="cve_id", right_on="cveID",
                   how="left", validate="many_to_one")       # a duplicate key now fails loudly (A36)
            .merge(epss[["cve", "epss", "percentile"]], left_on="cve_id", right_on="cve",
                   how="left", validate="many_to_one")
            .drop(columns=["cveID", "cve"])
        )
        assert len(merged) == len(nvd), "a join changed the row count"
        # The join may add exactly three columns. (KEV also has a `cwes` column - a name
        # check alone would confuse it with NVD's - so compare whole column sets.)
        extra = set(merged.columns) - set(nvd.columns)
        assert extra == {"dateAdded", "epss", "percentile"}, f"unexpected columns joined in: {extra}"

        merged["in_kev"] = merged["dateAdded"].notna()
        # Calendar days (A2). Negative = listed before the publication DAY; same day = 0.
        merged["days_to_kev"] = calendar_days(merged["dateAdded"], merged["published"])
        merged["kev_before_publication"] = merged["days_to_kev"] < 0          # B16

        live = merged[~merged["rejected"]]
        k = live[live["in_kev"]]
        n_kev, share = len(k), live["in_kev"].mean()
        rejected_in_kev = int(merged.loc[merged["rejected"], "in_kev"].sum())
        print(f"rows                   : {len(merged):,}  ({len(live):,} live)")
        print(f"in KEV (live)          : {n_kev} ({share:.3%})")
        print(f"Rejected but in KEV    : {rejected_in_kev}  (expected 0)")
        print(f"has EPSS score         : {live['epss'].notna().mean():.1%} of live records (all were queried)")
        print(f"has CNA CVSS score     : {1 - live['cvss_missing'].mean():.1%}")
        print(f"has CNA CWE            : {1 - live['cwe_missing'].mean():.1%}")

        timing = {}
        if n_kev:
            d = k["days_to_kev"]
            timing = {"listed_before_publication_day": int((d < 0).sum()),
                      "listed_on_or_before_publication_day": int((d <= 0).sum()),
                      "median_days": float(d.median()),
                      "within_30_days": int((d <= 30).sum()), "within_365_days": int((d <= 365).sum()),
                      "within_730_days": int((d <= 730).sum()), "n": n_kev}
            print(f"\nlisting timing (calendar days, A2): before the publication day "
                  f"{timing['listed_before_publication_day']}, on or before it "
                  f"{timing['listed_on_or_before_publication_day']}, median {timing['median_days']:.0f} days")
            print("\nexploited CVEs in this window:")
            cols = ["cve_id", "cna", "cvss_score", "cvss_version", "epss", "days_to_kev", "cwe_primary"]
            print(k[cols].sort_values("days_to_kev").to_string(index=False))

        # Figures the proposal quotes, recomputed on CNA-only scores (B8, C1, C6). A missing
        # CNA score counts as "not >= 7": a rule can only act on scores that exist.
        def rule(mask, name):
            caught = int((mask & live["in_kev"]).sum())
            return {"rule": name, "flagged": int(mask.sum()), "effort": float(mask.mean()),
                    "caught": caught, "coverage": caught / n_kev if n_kev else None,
                    "precision": caught / int(mask.sum()) if mask.sum() else None}
        rules = [rule(live["cvss_score"].fillna(0) >= 7, "CNA CVSS >= 7"),
                 rule(live["cna"] == "secure@microsoft.com", "CNA is Microsoft")]
        by_sev = (live.assign(sev=live["cvss_severity"].fillna("(no CNA score)"))
                      .groupby("sev")["in_kev"].agg(["size", "sum", "mean"]))
        print("\nexploitation rate by CNA severity (B1: 'no CNA score' is its own group):")
        print(by_sev.to_string(formatters={"mean": lambda x: f"{x:.3%}"}))
        print("\nsimple rules (effort = share of records flagged):")
        print(pd.DataFrame(rules).to_string(index=False))

        REPORT["findings"]["join"] = {
            "live": len(live), "in_kev": n_kev, "share": float(share), "rejected_in_kev": rejected_in_kev,
            "timing": timing, "rules": rules,
            "rate_by_severity": {str(i): {"n": int(r["size"]), "kev": int(r["sum"])} for i, r in by_sev.iterrows()},
        }
        # 0 hits is possible in a quiet window (WARN); thousands would mean a wrong key (A37).
        ok = None if n_kev == 0 else (share < 0.02 and rejected_in_kev == 0)
        record("14-join", ok, rows=len(merged), in_kev=n_kev, share=round(float(share), 5))
    except (AssertionError, pd.errors.MergeError, KeyError) as exc:
        merged = pd.DataFrame()
        record("14-join", False, error=repr(exc))

rows                   : 10,182  (9,302 live)
in KEV (live)          : 51 (0.548%)
Rejected but in KEV    : 0  (expected 0)
has EPSS score         : 100.0% of live records (all were queried)
has CNA CVSS score     : 59.1%
has CNA CWE            : 55.8%

listing timing (calendar days, A2): before the publication day 4, on or before it 13, median 20 days

exploited CVEs in this window:
        cve_id                                         cna  cvss_score cvss_version    epss  days_to_kev cwe_primary
 CVE-2019-8720                         secalert@redhat.com         NaN          NaN 0.01556       -287.0     CWE-119
CVE-2022-42475                          psirt@fortinet.com         9.8          3.1 0.99474        -20.0     CWE-197
CVE-2023-23529                  product-security@apple.com         NaN          NaN 0.09426        -13.0         NaN
CVE-2023-26360                             psirt@adobe.com         8.6          3.1 0.97339         -8.0     CWE-284
CVE-2023-21715              

#### Reference: what the numbers should look like

| Check | Expected | If it is off |
|:--|:--|:--|
| NVD `totalResults` | ~400,000 | Wrong endpoint, or an unintended filter |
| 10-request burst | all `200` | Key not applied — check the `apiKey` header |
| Same burst without the key | some 4xx refusals | None at all: step 06 proves nothing |
| KEV count | ~1,700–1,800 | Feed moved; find the current URL on cisa.gov |
| EPSS total | ~380,000 | Fine — just record what you saw |
| CVEs in one window | ~10,000, equal to `totalResults` | Exactly 2,000 means paging stopped early |
| In KEV | a few dozen (~0.5%) | Zero is possible; thousands means a wrong join key |

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 15 &nbsp;&middot;&nbsp; CONSTRAINTS</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Are window edges inclusive?</div>
</div>


**Why** — consecutive windows share a boundary: one ends at the exact moment the next begins.
Whether NVD includes a record published *exactly* on an edge decides whether the full download
repeats it, or misses it.

**How** — take one record's publication timestamp, to the millisecond. Ask for a window that
*starts* there, and one that *ends* there, and see which contains it.

**Pass** — at least one end is inclusive, so nothing is missed. If both are, the full download
removes duplicates on `cve_id`. **WARN** if neither is.

In [19]:
def iso_ms(ts):
    """A timestamp in NVD's query format, to the millisecond."""
    return ts.strftime("%Y-%m-%dT%H:%M:%S.") + f"{ts.microsecond // 1000:03d}Z"

try:
    if nvd.empty:
        raise ApiError("needs the window from step 11")
    probe = nvd[~nvd["rejected"]].iloc[len(nvd) // 2]          # any live record will do
    t, day = probe["published"], pd.Timedelta(days=1)
    print(f"probe record: {probe['cve_id']}, published {iso_ms(t)}")
    from_t = {c["id"] for c in client.iter_cves(progress=False, pubStartDate=iso_ms(t),
                                                pubEndDate=iso_ms(t + day))}
    to_t = {c["id"] for c in client.iter_cves(progress=False, pubStartDate=iso_ms(t - day),
                                              pubEndDate=iso_ms(t))}
    start_incl, end_incl = probe["cve_id"] in from_t, probe["cve_id"] in to_t
    print(f"a window STARTING at exactly that moment includes it: {start_incl}")
    print(f"a window ENDING at exactly that moment includes it  : {end_incl}")
    if start_incl and end_incl:
        meaning = ("both ends inclusive: a record on a shared boundary is fetched twice, "
                   "so the full pull must de-duplicate on cve_id (A10)")
    elif start_incl or end_incl:
        meaning = "one end inclusive: adjacent windows neither miss nor repeat a record"
    else:
        meaning = "neither end inclusive: a record exactly on a boundary would be MISSED"
    print("\nmeaning:", meaning)
    REPORT["findings"]["window_boundaries"] = {"start_inclusive": start_incl,
                                               "end_inclusive": end_incl, "meaning": meaning}
    # Missing records is the dangerous case; duplicates are handled by de-duplication.
    record("15-window-boundaries", (start_incl or end_incl) or None,
           start_inclusive=start_incl, end_inclusive=end_incl)
except (ApiError, ValueError, KeyError) as exc:
    record("15-window-boundaries", False, error=repr(exc))

probe record: CVE-2023-1328, published 2023-03-10T17:15:09.973Z
a window STARTING at exactly that moment includes it: True
a window ENDING at exactly that moment includes it  : False

meaning: one end inclusive: adjacent windows neither miss nor repeat a record
PASS  15-window-boundaries  |  {'start_inclusive': True, 'end_inclusive': False}


<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 16 &nbsp;&middot;&nbsp; PROVENANCE</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Can we see a record as it was at publication?</div>
</div>


**Why** — the API returns each record *as it is today*. Descriptions, scores and CWEs can all be
edited after publication, and an edit that mentions exploitation would leak the answer.

**Two probes**

| Probe | Question |
|:--|:--|
| **NVD CVE Change History API** (3 exploited CVEs) | Does it record *what* changed, *when*, and *who* changed it — enough to rebuild the publication-time record? |
| **CVE Services API** (about 20 CVEs) | How close is NVD's `published` timestamp to the CVE Program's own publication time? |

The raw change-history responses are saved in `logs/probes/` for a closer read.

**Pass** — both services answer, and NVD's publication time is within a day of the CVE Program's
for every record checked. Larger gaps are a **WARN**, listed for review.

In [20]:
HISTORY_URL = "https://services.nvd.nist.gov/rest/json/cvehistory/2.0"
history_summary = []
try:
    if merged.empty:
        raise ApiError("needs the joined table from step 14")
    lk = merged[~merged["rejected"] & merged["in_kev"]]
    # Three exploited CVEs: Apple's (its description mentions exploitation, B9), one carrying
    # a CISA-ADP score (A1), and the earliest-published one.
    picks = list(dict.fromkeys(
        [c for c in ["CVE-2023-23529"] if c in set(lk["cve_id"])]
        + lk.loc[lk["diag_adp_score"].notna(), "cve_id"].head(1).tolist()
        + lk.sort_values("published")["cve_id"].head(3).tolist()))[:3]
    for cve_id in picks:
        body = http_get(HISTORY_URL, params={"cveId": cve_id}, headers=NVD_HEADERS,
                        pause=client.pause).json()
        (PROBES / f"cvehistory_{cve_id}.json").write_text(json.dumps(body, indent=1))
        changes = [c.get("change", {}) for c in body.get("cveChanges", [])]
        details = [d for c in changes for d in c.get("details", [])]
        created = sorted(c.get("created", "") for c in changes if c.get("created"))
        history_summary.append({
            "cve_id": cve_id, "events": len(changes),
            "first_event": created[0] if created else None,
            "last_event": created[-1] if created else None,
            "event_names": dict(Counter(c.get("eventName") for c in changes)),
            "changed_by": sorted({c.get("sourceIdentifier") for c in changes if c.get("sourceIdentifier")}),
            "detail_types": dict(Counter(d.get("type") for d in details).most_common(10)),
            "description_details": sum(1 for d in details if "description" in str(d.get("type", "")).lower()),
        })
    note_source("NVD CVE Change History API 2.0", cves=picks)
    for s in history_summary:
        print(f"{s['cve_id']}: {s['events']} events, {s['first_event']} -> {s['last_event']}")
        print(f"   events     : {s['event_names']}")
        print(f"   changed by : {s['changed_by']}")
        print(f"   what       : {s['detail_types']}")
        print(f"   description changes recorded: {s['description_details']}")
    print(f"\nraw responses saved in {PROBES} for a closer read (V19)")
    REPORT["findings"]["change_history"] = history_summary
    record("16-change-history", bool(history_summary) and all(s["events"] > 0 for s in history_summary),
           cves=picks)
except (ApiError, ValueError, KeyError) as exc:
    record("16-change-history", False, error=repr(exc))

CVE-2023-23529: 17 events, 2023-03-08T15:55:03.213 -> 2026-06-17T05:37:21.160
   events     : {'Initial Analysis': 1, 'CVE Modified': 13, 'Modified Analysis': 3}
   changed by : ['134c704f-9b21-4f2e-91b3-4a467353bcc0', 'af854a3a-2127-422b-91ae-364da2661108', 'nvd@nist.gov', 'product-security@apple.com']
   what       : {'Reference': 14, 'Reference Type': 13, 'CVSS V3.1': 2, 'CWE': 2, 'CPE Configuration': 2, 'Description': 2, 'Affected': 1, 'SSVC': 1}
   description changes recorded: 2
CVE-2022-44877: 19 events, 2023-01-06T17:15:09.727 -> 2026-06-17T05:09:01.120
   events     : {'CVE Modified': 13, 'Initial Analysis': 1, 'CPE Deprecation Remap': 1, 'Modified Analysis': 4}
   changed by : ['134c704f-9b21-4f2e-91b3-4a467353bcc0', 'af854a3a-2127-422b-91ae-364da2661108', 'cve@mitre.org', 'nvd@nist.gov']
   what       : {'Reference': 13, 'Reference Type': 7, 'CVSS V3.1': 2, 'CWE': 2, 'CPE Configuration': 2, 'Description': 1, 'Affected': 1, 'SSVC': 1}
   description changes recorded: 1
CVE-20

<div style="background:#e5e2ee;border-left:4px solid #9a7b2f;padding:8px 13px;border-radius:2px;">
  <div style="font:700 10.5px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.11em;color:#9a7b2f;">WHY THE PUBLICATION TIMESTAMP MATTERS</div>
  <div style="font:400 13.5px/1.55 system-ui,-apple-system,'Segoe UI',sans-serif;color:#2a2a2a;margin-top:4px;">&ldquo;What was known at publication&rdquo; is the project's central rule, so the publication moment has to be right. In the October 1 sample, one exploited CVE (CVE-2019-8720) was in KEV 288 days <i>before</i> NVD's publication date &mdash; this probe shows whether NVD's date lags the CVE Program's.</div>
</div>

In [21]:
CVE_SERVICES_URL = "https://cveawg.mitre.org/api/cve/"
try:
    if merged.empty:
        raise ApiError("needs the joined table from step 14")
    lm = merged[~merged["rejected"]]
    # Exploited CVEs first (those listed before publication lead), then 10 random others.
    ids = list(dict.fromkeys(
        lm.loc[lm["kev_before_publication"], "cve_id"].tolist()
        + lm.loc[lm["in_kev"], "cve_id"].head(10).tolist()
        + lm.loc[~lm["in_kev"], "cve_id"].sample(min(10, int((~lm["in_kev"]).sum())), random_state=510).tolist()))
    rows = []
    for cve_id in ids:
        try:
            meta = http_get(CVE_SERVICES_URL + cve_id, pause=0.5).json().get("cveMetadata", {})
            rows.append({"cve_id": cve_id, "program_published": meta.get("datePublished"),
                         "assigner": meta.get("assignerShortName")})
        except ApiError as exc:
            rows.append({"cve_id": cve_id, "program_published": None, "assigner": repr(exc)[:60]})
    note_source("CVE Services API (CVE Program)", records=len(rows))
    cmp_ = pd.DataFrame(rows).merge(lm[["cve_id", "published", "in_kev"]], on="cve_id")
    cmp_["program_published"] = to_naive_utc(cmp_["program_published"])
    cmp_["nvd_minus_program_h"] = ((cmp_["published"] - cmp_["program_published"])
                                   .dt.total_seconds() / 3600).round(1)
    print(cmp_.to_string(index=False))
    h = cmp_["nvd_minus_program_h"].dropna()
    print(f"\nNVD 'published' minus CVE Program 'datePublished', hours: median {h.median():.1f}, "
          f"largest gap {h.abs().max():.1f}  (n={len(h)})")
    REPORT["findings"]["nvd_vs_cve_program_hours"] = {
        "n": int(len(h)), "median": float(h.median()) if len(h) else None,
        "max_abs": float(h.abs().max()) if len(h) else None,
        "rows": cmp_.astype(str).to_dict("records")}
    # PASS = measured and close (within a day); WARN = measured but some records differ a lot.
    ok = (h.abs() <= 24).all() or None
    complete = len(h) == len(cmp_)
    record("16-cve-program-dates", ok if complete else (None if len(h) >= 0.8 * len(cmp_) else False),
           measured=int(len(h)), of=len(cmp_),
           over_24h=int((h.abs() > 24).sum()))
except (ApiError, ValueError, KeyError) as exc:
    record("16-cve-program-dates", False, error=repr(exc))

        cve_id       program_published  assigner               published  in_kev  nvd_minus_program_h
CVE-2022-42475 2023-01-02 08:18:49.444  fortinet 2023-01-02 09:15:09.490    True                  0.9
CVE-2023-23529 2023-02-27 00:00:00.000     apple 2023-02-27 20:15:14.710    True                 20.3
 CVE-2019-8720 2023-03-06 00:00:00.000    redhat 2023-03-06 23:15:10.287    True                 23.3
CVE-2023-26360 2023-03-23 00:00:00.000     adobe 2023-03-23 20:15:15.263    True                 20.3
CVE-2022-44877 2023-01-05 00:00:00.000     mitre 2023-01-05 23:15:09.150    True                 23.3
CVE-2023-21674 2023-01-10 00:00:00.000 microsoft 2023-01-10 22:15:16.307    True                 22.3
CVE-2023-22952 2023-01-11 00:00:00.000     mitre 2023-01-11 09:15:08.787    True                  9.3
CVE-2023-21839 2023-01-17 23:35:09.774    oracle 2023-01-18 00:15:13.450    True                  0.7
CVE-2022-47966 2023-01-18 00:00:00.000     mitre 2023-01-18 18:15:10.570    True  

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 17 &nbsp;&middot;&nbsp; SCOPE</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">How big is the dataset, and are there enough positives?</div>
</div>


**Why** — two decisions depend on numbers we have not measured yet:

1. **Dataset size** (2016–2024), counted directly instead of estimated. This takes about 60
   tiny count requests (a couple of minutes).
2. **The modelling window.** KEV began on **Nov 3, 2021**. A CVE published before about November 2020
   *could not* be listed within 365 days of publication. So the model uses only CVEs published
   from the catalog's launch onward:

| Split | Published | Role |
|:--|:--|:--|
| pre-KEV | 2016 – Nov 2, 2021 | Description only (labelled "ever listed") |
| train | Nov 3, 2021 – 2022 | Model fitting |
| validate | 2023 | Tuning |
| test | 2024 | The final, untouched evaluation |

Splits always use the `published` date, never the year in the CVE ID.

In [22]:
KEV_LAUNCH = dt.datetime(2021, 11, 3)                 # the catalog's first day (step 12)
SPLITS = {                                            # B13 option (a); split on `published` (B11)
    "pre-KEV (description only)": (dt.datetime(2016, 1, 1), KEV_LAUNCH),
    "train":                      (KEV_LAUNCH, dt.datetime(2023, 1, 1)),
    "validate":                   (dt.datetime(2023, 1, 1), dt.datetime(2024, 1, 1)),
    "test":                       (dt.datetime(2024, 1, 1), dt.datetime(2025, 1, 1)),
}
# Count queries return one record each, so a shorter pause is used: 1.2 s is about
# 25 requests per 30 s, half the keyed limit. The full download keeps the 6 s pause.
counter = NVDClient(NVD_API_KEY, pause=1.2)
sizes = pd.DataFrame()
try:
    size_rows = []
    for name, (a, b) in SPLITS.items():
        pairs = list(counter.windows_between(a, b))
        assert not check_windows(pairs, a, b)[0], f"bad windows for {name}"
        n_all = n_live = 0
        for s, e in pairs:
            n_all += counter.count(pubStartDate=s, pubEndDate=e)
            n_live += counter.count(flags=["noRejected"], pubStartDate=s, pubEndDate=e)
        size_rows.append({"split": name, "from": f"{a:%Y-%m-%d}", "to (excl.)": f"{b:%Y-%m-%d}",
                          "windows": len(pairs), "all_records": n_all, "excluding_rejected": n_live})
        print(f"  {name:<28} {n_all:>8,} records, {n_live:>8,} excluding Rejected")
    sizes = pd.DataFrame(size_rows)
    note_source("NVD CVE API 2.0 (counts)", queries=int(2 * sizes["windows"].sum()))
    total_all, total_live = int(sizes["all_records"].sum()), int(sizes["excluding_rejected"].sum())
    print(f"\n2016-2024 total: {total_all:,} records ({total_live:,} excluding Rejected)  <- V12, C4")
    print(f"full download at 2,000 per page and a 6 s pause: about {total_all // 2000 + 30} "
          f"requests, roughly {(total_all // 2000 + 30) * 8 / 60:.0f} minutes  <- C46")
    flag_works = (sizes["excluding_rejected"] < sizes["all_records"]).any()
    if not flag_works:
        print("  CHECK: excluding Rejected changed nothing - NVD may have ignored the noRejected flag")
    print("  note: if both window ends are inclusive (step 15), boundary records are counted twice;"
          " that is at most a handful")
    REPORT["findings"]["dataset_size"] = sizes.to_dict("records")
    record("17-dataset-size", bool(flag_works), total_2016_2024=total_all,
           excluding_rejected=total_live)
except (ApiError, ValueError, KeyError, AssertionError) as exc:
    record("17-dataset-size", False, error=repr(exc))
sizes

  pre-KEV (description only)     99,009 records,   90,082 excluding Rejected
  train                          30,316 records,   28,368 excluding Rejected
  validate                       30,949 records,   28,816 excluding Rejected
  test                           40,704 records,   39,953 excluding Rejected

2016-2024 total: 200,978 records (187,219 excluding Rejected)  <- V12, C4
full download at 2,000 per page and a 6 s pause: about 130 requests, roughly 17 minutes  <- C46
  note: if both window ends are inclusive (step 15), boundary records are counted twice; that is at most a handful
PASS  17-dataset-size  |  {'total_2016_2024': 200978, 'excluding_rejected': 187219}


,split,from,to (excl.),windows,all_records,excluding_rejected
0,pre-KEV (description only),2016-01-01,2021-11-03,18,99009,90082
1,train,2021-11-03,2023-01-01,4,30316,28368
2,validate,2023-01-01,2024-01-01,4,30949,28816
3,test,2024-01-01,2025-01-01,4,40704,39953


**Label feasibility** — one download of NVD's record for every KEV CVE (via the `hasKev` filter)
gives each one's publication date. Positives are then counted per split under the 365-day label.

**Pass** — at least **100** training positives (a working threshold: about 10 positives per predictor
for a ~10-predictor logistic regression). The cell also confirms the KEV snapshot reaches far enough
past 2024 for the test year's 365-day window.

In [23]:
MIN_TRAIN_POSITIVES = 100     # working threshold: ~10 positives per predictor for a ~10-predictor
                              # logistic regression (the usual events-per-variable rule of thumb)
kev_dates = pd.DataFrame()
try:
    if kev.empty:
        raise ApiError("needs the KEV catalog from step 12")
    print("downloading NVD's record of every KEV CVE (its publication date is what we need) ...")
    recs = [{"cve_id": c["id"], "published": c.get("published"),
             "nvd_kev_added": c.get("cisaExploitAdd"), "rejected": c.get("vulnStatus") == "Rejected"}
            for c in client.iter_cves(flags=["hasKev"])]
    note_source("NVD CVE API 2.0 (hasKev)", records=len(recs), **client.last_run)
    nk = pd.DataFrame(recs)
    nk["published"] = to_naive_utc(nk["published"])
    nk["nvd_kev_added"] = to_naive_utc(nk["nvd_kev_added"])
    kev_dates = nk.merge(kev[["cveID", "dateAdded"]], left_on="cve_id", right_on="cveID",
                         how="outer", validate="one_to_one", indicator="found_in")
    kev_dates["found_in"] = kev_dates["found_in"].astype(str).map(
        {"both": "both", "left_only": "NVD only", "right_only": "KEV only"})
    print(kev_dates["found_in"].value_counts().to_string())
    kd = kev_dates[kev_dates["found_in"] == "both"].copy()
    kd["days_to_kev"] = calendar_days(kd["dateAdded"], kd["published"])
    disagree = int((kd["nvd_kev_added"].notna()
                    & (kd["nvd_kev_added"].dt.normalize() != kd["dateAdded"])).sum())
    print(f"NVD's copy of the KEV date disagrees with KEV's own: {disagree}")

    live_counts = sizes.set_index("split")["excluding_rejected"] if not sizes.empty else pd.Series(dtype=float)
    rows = []
    for name, (a, b) in SPLITS.items():
        in_split = kd["published"].ge(a) & kd["published"].lt(b) & ~kd["rejected"]
        within = in_split & (kd["days_to_kev"] <= 365)
        n = live_counts.get(name)
        rows.append({"split": name, "records": n,
                     "ever_listed": int(in_split.sum()),
                     "listed_within_365d": int(within.sum()),
                     "of_which_before_publication": int((within & (kd["days_to_kev"] < 0)).sum()),
                     "base_rate_365d": round(int(within.sum()) / n, 5) if n else None})
    feas = pd.DataFrame(rows)
    print("\npositives per split under the 365-day label (B13 option a):")
    print(feas.to_string(index=False))

    pre = feas.iloc[0]
    print(f"\npre-KEV CVEs: {pre['ever_listed']} were listed eventually, but only "
          f"{pre['listed_within_365d']} within 365 days of publication - why B13 matters.")
    snapshot, needed = kev["dateAdded"].max(), dt.datetime(2025, 12, 31)
    print(f"KEV snapshot runs to {snapshot:%Y-%m-%d}; the test year's 365-day window needs "
          f"{needed:%Y-%m-%d}: {'covered' if snapshot >= needed else 'NOT covered'}")

    train_pos = int(feas.loc[feas["split"] == "train", "listed_within_365d"].iloc[0])
    enough = train_pos >= MIN_TRAIN_POSITIVES
    print(f"\ntraining positives: {train_pos}  (working threshold {MIN_TRAIN_POSITIVES}) -> "
          + ("enough to proceed with option (a)" if enough else "TOO FEW: revisit B13"))
    REPORT["findings"]["label_feasibility"] = {
        "splits": feas.to_dict("records"), "kev_found_in": kev_dates["found_in"].value_counts().to_dict(),
        "nvd_kev_date_disagreements": disagree, "train_positives": train_pos,
        "threshold": MIN_TRAIN_POSITIVES, "snapshot_covers_test": bool(snapshot >= needed)}
    record("17-label-feasibility", enough or None, train_positives=train_pos,
           validate_positives=int(feas.loc[feas["split"] == "validate", "listed_within_365d"].iloc[0]),
           test_positives=int(feas.loc[feas["split"] == "test", "listed_within_365d"].iloc[0]))
except (ApiError, ValueError, KeyError, pd.errors.MergeError) as exc:
    record("17-label-feasibility", False, error=repr(exc))

downloading NVD's record of every KEV CVE (its publication date is what we need) ...
  1,773/1,773
found_in
both    1773
NVD's copy of the KEV date disagrees with KEV's own: 0

positives per split under the 365-day label (B13 option a):
                     split  records  ever_listed  listed_within_365d  of_which_before_publication  base_rate_365d
pre-KEV (description only)    90082          683                 159                            0         0.00177
                     train    28368          157                 128                           35         0.00451
                  validate    28816          166                 145                           13         0.00503
                      test    39953          161                 149                            7         0.00373

pre-KEV CVEs: 683 were listed eventually, but only 159 within 365 days of publication - why B13 matters.
KEV snapshot runs to 2026-11-09; the test year's 365-day window needs 2025-12-31: cover

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 18 &nbsp;&middot;&nbsp; EVIDENCE</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">Save the sample, and log the access dates</div>
</div>


**Why** — three reasons, all of which matter later:

1. **Access dates per source are a graded requirement** — stamped when each source's data actually
   arrived, with the source's own version marker (KEV catalog version, EPSS score date).
2. **The exact environment is recorded** — library versions in the report, and every package pinned
   in `requirements-lock.txt`.
3. The report is **machine-readable**, so the run can be diagnosed without pasting screenshots. A dated
   copy (`logs/validation_report_YYYY-MM-DD.json`) is kept beside it, so step 19 can compare runs.

**Pass** — the files are written, and the summary lists every expected step. A step that never ran
shows as FAIL, so a partial run can't look complete.

In [24]:
def save(df, path):
    """Write parquet, which keeps dates and types; skip empty tables."""
    if df.empty:
        print(f"  skipped {path} (empty)")
        return None
    df.to_parquet(path, index=False)
    print(f"  {path}  ({path.stat().st_size / 1024:.0f} KB)")
    return path.as_posix()

print("saved:")
files = [save(merged, PROC / "sample_merged.parquet"),
         save(kev, RAW / "kev_catalog.parquet"),
         save(nvd, RAW / "nvd_sample_window.parquet"),
         save(kev_dates, PROC / "kev_publication_dates.parquet")]

# The exact environment that produced these results (A38).
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
if freeze.returncode == 0 and freeze.stdout.strip():
    Path("requirements-lock.txt").write_text(freeze.stdout)
    files.append("requirements-lock.txt")
    print("  requirements-lock.txt  (exact package versions)")

REPORT["files"] = [f for f in files if f]
REPORT["versions"] = {"python": sys.version.split()[0], "pandas": pd.__version__,
                      "numpy": np.__version__, "requests": requests.__version__,
                      "pyarrow": pyarrow.__version__}
record("18-saved", len(REPORT["files"]) >= 3, files=len(REPORT["files"]))

def to_json(obj):
    """numpy numbers become plain numbers; anything else unusual becomes text."""
    return obj.item() if isinstance(obj, np.generic) else str(obj)

def write_report(pending=()):
    """Write the report twice - the latest copy, and a dated copy kept so that a later run can be
    compared with this one (step 19) - then print the tally. A step that never ran is a FAIL, not
    an absence (A40); steps still to come in this run are listed in `pending` instead."""
    for step in EXPECTED_STEPS:
        if step not in pending:
            REPORT["steps"].setdefault(step, {"ok": False, "verdict": "FAIL", "error": "step never ran"})
    REPORT["finished"] = now_iso()
    text = json.dumps(REPORT, indent=2, default=to_json)
    latest, dated = LOGS / "validation_report.json", LOGS / f"validation_report_{REPORT['started'][:10]}.json"
    latest.write_text(text)
    dated.write_text(text)
    done = [s for s in EXPECTED_STEPS if s not in pending]
    tally = Counter(REPORT["steps"][s]["verdict"] for s in done)
    print(f"\n{tally['PASS']}/{len(done)} checks passed, {tally['WARN']} to review, {tally['FAIL']} failed"
          + (f"  ({', '.join(pending)} still to run)" if pending else ""))
    for step in done:
        r = REPORT["steps"][step]
        why = r.get("error") or (f"failed: {r['failed']}" if r.get("failed") else "")
        print(f"  {r['verdict']:<5} {step}" + (f"   <- {why}" if why and r["verdict"] != "PASS" else ""))
    print("\naccess dates:")
    for name, s in REPORT["sources"].items():
        print(f"  {name:<34} {s['accessed']}")
    print(f"\nfull report: {latest}  (dated copy: {dated.name})")

write_report(pending=["19-compare-runs"])


saved:
  data\processed\sample_merged.parquet  (1537 KB)
  data\raw\kev_catalog.parquet  (281 KB)
  data\raw\nvd_sample_window.parquet  (1430 KB)
  data\processed\kev_publication_dates.parquet  (58 KB)
  requirements-lock.txt  (exact package versions)
PASS  18-saved  |  {'files': 5}

18/19 checks passed, 0 to review, 1 failed  (19-compare-runs still to run)
  PASS  03-key-loaded
  PASS  05-nvd-reachable
  PASS  06-key-honoured
  FAIL  06b-negative-control
  PASS  07-known-record
  PASS  08-date-limit-understood
  PASS  09-client-built
  PASS  10-parser-selftest
  PASS  11-nvd-window
  PASS  12-kev
  PASS  13-epss
  PASS  13b-epss-bulk
  PASS  14-join
  PASS  15-window-boundaries
  PASS  16-change-history
  PASS  16-cve-program-dates
  PASS  17-dataset-size
  PASS  17-label-feasibility
  PASS  18-saved

access dates:
  NVD CVE API 2.0                    2026-11-10T00:21:21-08:00
  CISA KEV catalog                   2026-11-10T00:21:23-08:00
  EPSS (FIRST)                       2026-11-1

<div style="background:#e5e2ee;border-left:5px solid #8a3324;padding:9px 14px;border-radius:2px;">
  <div style="font:700 11px/1.4 system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:.12em;color:#8a3324;">STEP 19 &nbsp;&middot;&nbsp; DRIFT</div>
  <div style="font:600 17px/1.35 system-ui,-apple-system,'Segoe UI',sans-serif;color:#1b1b1b;margin-top:3px;">What changed since the last run?</div>
</div>


**Why** — the sources are live, and they change. Some numbers *should* grow (the catalogues), some
should **never move** (anything about a time window that has already closed), and a few may drift
slightly as records are edited or withdrawn. Comparing against the previous run separates normal growth
from a change that would affect the project.

| Expectation | Figures | If it doesn't hold |
|:--|:--|:--|
| **same** | Positives per split under the 365-day label, KEV launch-day entries, historical EPSS scores, window-edge behaviour | History was revised: the labels or the benchmark need a second look |
| **grows** | NVD total, KEV entries, EPSS coverage, CVEs ever listed | A catalogue that shrinks is unusual and worth reading |
| **may drift** | Records in the sample window, CNA score share, 2016–2024 counts | A change above 2% deserves a look |

**How** — finds the most recent *earlier* report in `logs/` (the dated copies step 18 writes) and
compares figure by figure. The table is also saved as a CSV in `logs/`.

**Pass** — every *same* figure unchanged, nothing that should grow has shrunk, and drift within 2%.
Anything else is a **WARN**: the APIs work, but a person should read what changed.


In [25]:
# STEP 19: compare this run with the previous one.
def dig(report, path):
    """Follow a dotted path through a report. In a list, 'key=value' picks the matching item."""
    node = report
    for part in path.split("."):
        if isinstance(node, list):
            key, _, value = part.partition("=")
            node = next((x for x in node if isinstance(x, dict) and str(x.get(key)) == value), None)
        elif isinstance(node, dict):
            node = node.get(part)
        else:
            return None
    return node

PRE = "findings.label_feasibility.splits.split=pre-KEV (description only)"
COMPARE = [
    # label,                                      path in the report,                                  expectation
    ("NVD records, whole catalogue",              "steps.05-nvd-reachable.total_results",              "grows"),
    ("KEV catalogue entries",                     "steps.12-kev.entries",                              "grows"),
    ("EPSS CVEs scored",                          "steps.13-epss.total_scored",                        "grows"),
    ("KEV entries dated to its first day",        "steps.12-kev.launch_day_entries",                   "same"),
    ("Train positives (365-day label)",           "steps.17-label-feasibility.train_positives",        "same"),
    ("Validate positives (365-day label)",        "steps.17-label-feasibility.validate_positives",     "same"),
    ("Test positives (365-day label)",            "steps.17-label-feasibility.test_positives",         "same"),
    ("Pre-KEV CVEs listed within 365 days",       f"{PRE}.listed_within_365d",                         "same"),
    ("Pre-KEV CVEs ever listed",                  f"{PRE}.ever_listed",                                "grows"),
    ("Window start is inclusive",                 "findings.window_boundaries.start_inclusive",        "same"),
    ("Window end is inclusive",                   "findings.window_boundaries.end_inclusive",          "same"),
    ("Sample window: records",                    "steps.11-nvd-window.rows",                          "may drift"),
    ("Sample window: live records",               "findings.window_provenance.live",                   "may drift"),
    ("Sample window: share with a CNA score",     "findings.window_provenance.cna_score_share",        "may drift"),
    ("Sample window: CVEs ever in KEV",           "findings.join.in_kev",                              "grows"),
    ("CVEs published 2016-2024",                  "steps.17-dataset-size.total_2016_2024",             "may drift"),
    ("CVEs published 2016-2024, live",            "steps.17-dataset-size.excluding_rejected",          "may drift"),
]

def judge(before, now, expectation, tolerance=0.02):
    """Status of one figure: 'ok' or a short reason it needs a look."""
    if before is None or now is None:
        return "not in both runs"
    if expectation == "same":
        same = abs(float(before) - float(now)) < 1e-9 if isinstance(before, (int, float)) else before == now
        return "ok" if same else "CHANGED"
    if expectation == "grows":
        return "ok" if float(now) >= float(before) else "SHRANK"
    change = abs(float(now) - float(before)) / abs(float(before)) if float(before) else 0.0
    return "ok" if change <= tolerance else f"moved {change:.1%}"

previous, prev_path = None, None
for path in sorted(LOGS.glob("validation_report_*.json"), reverse=True):
    data = json.loads(path.read_text())
    if data.get("started") != REPORT["started"]:          # skip this run's own dated copy
        previous, prev_path = data, path
        break

if previous is None:
    record("19-compare-runs", None, error="no earlier report in logs/ to compare with")
else:
    checks = list(COMPARE)
    # Historical EPSS scores for the probe CVEs: a past day's score should never change.
    for row in previous["findings"].get("epss_historical_probe") or []:
        if isinstance(row, dict) and row.get("cve_id"):
            checks.append((f"EPSS the day after publication, {row['cve_id']}",
                           f"findings.epss_historical_probe.cve_id={row['cve_id']}.epss_then", "same"))
    rows = []
    for label, path, expectation in checks:
        before, now = dig(previous, path), dig(REPORT, path)
        status = judge(before, now, expectation)
        delta = (float(now) - float(before)
                 if isinstance(before, (int, float)) and isinstance(now, (int, float))
                 and not isinstance(before, bool) else None)
        rows.append({"figure": label, "before": before, "now": now,
                     "change": None if delta is None else round(delta, 6),
                     "expected": expectation, "status": status})
    comparison = pd.DataFrame(rows)
    before_day = prev_path.stem.replace("validation_report_", "")      # the dated copy's own date
    now_day = REPORT["started"][:10]
    print(f"comparing with {prev_path.name}  ({before_day} -> {now_day})\n")
    print(comparison.to_string(index=False))
    out = LOGS / f"validation_comparison_{before_day}_vs_{now_day}.csv"
    comparison.to_csv(out, index=False)
    print(f"\nsaved: {out}")

    flagged = comparison[comparison["status"] != "ok"]
    REPORT["findings"]["comparison"] = {"previous_report": prev_path.name, "previous_started": previous["started"],
                                        "rows": comparison.to_dict("records")}
    record("19-compare-runs", flagged.empty or None, compared_with=prev_path.name,
           figures=len(comparison), flagged=flagged["figure"].tolist())

write_report()


comparing with validation_report_2026-10-09.json  (2026-10-09 -> 2026-11-10)

                                        figure    before       now  change  expected status
                  NVD records, whole catalogue    403391    421911 18520.0     grows     ok
                         KEV catalogue entries      1739      1773    34.0     grows     ok
                              EPSS CVEs scored    384534    402199 17665.0     grows     ok
            KEV entries dated to its first day       287       287     0.0      same     ok
               Train positives (365-day label)       128       128     0.0      same     ok
            Validate positives (365-day label)       145       145     0.0      same     ok
                Test positives (365-day label)       149       149     0.0      same     ok
           Pre-KEV CVEs listed within 365 days       159       159     0.0      same     ok
                      Pre-KEV CVEs ever listed       683       683     0.0     grows     ok
  

---

## Next steps once every check passes

| # | Step | Depends on |
|:--|:--|:--|
| 1 | **Confirm the modelling window** from step 17's positives per split | Step 17 |
| 2 | **Run the full download** for the windows step 17 counted — the request count and time follow from its totals. Write one file per window, so a failure costs one window, not the whole download; de-duplicate on `cve_id` if step 15 found both edges inclusive. | Steps 9, 11, 15, 17 |
| 3 | **Build the label** — listed in KEV within 365 days of publication (calendar days), for CVEs published from Nov 3, 2021 | Steps 12, 14, 17 |
| 4 | **Start the analysis** — questions Q1 and Q3 first, since they stand on their own | all |
| 5 | **Collect the EPSS benchmark** — the daily files from January 2, 2024 to January 31, 2025, keeping only the test-set CVEs' rows | Steps 13, 13b |

Keep this notebook in the repo as `00_validate_sources.ipynb`. It is the reproducibility evidence
for the data-acquisition section of the final report.